# Setup

## Installations

In [ ]:
# ============================================================
# RUN THIS CELL FIRST IN EVERY NEW COLAB RUNTIME
#
# Environment:
#   Python 3.13
#   PyTorch 2.11.0
#   Torchvision 0.26.0
#   CUDA 12.8
# ============================================================

import platform
import sys

# Fail early with a useful message if Colab changes Python versions.
if sys.version_info[:2] != (3, 13):
    raise RuntimeError(
        "This setup is pinned for Python 3.13, but this runtime is using "
        f"Python {platform.python_version()}."
    )

# Git is required to install the pinned torch-fidelity revision.
!command -v git >/dev/null || (apt-get -qq update && apt-get -qq install -y git)

# Keep the build tools compatible with the older packages used here.
%pip install -q --upgrade \
    "pip>=24.1,<26" \
    "setuptools<81" \
    wheel

# ------------------------------------------------------------
# 1. Install the EXACT PyTorch / CUDA 12.8 stack
#
# Important:
# Colab may already contain torch 2.11.0+cu130.
# "torch==2.11.0" considers that version compatible, so explicitly
# request +cu128 and force reinstall it.
# ------------------------------------------------------------

%pip install -q --upgrade --force-reinstall --no-cache-dir \
    "torch==2.11.0+cu128" \
    "torchvision==0.26.0+cu128" \
    "torchaudio==2.11.0+cu128" \
    --index-url https://download.pytorch.org/whl/cu128

# ------------------------------------------------------------
# 2. Install the ordinary Python dependencies
# ------------------------------------------------------------

%pip install -q --upgrade \
    numpy==2.1.3 \
    scipy==1.15.3 \
    packaging==24.2 \
    opencv-python==4.10.0.84 \
    timm==1.0.8 \
    wandb==0.30.0 \
    lovely-tensors==0.1.16 \
    einops==0.8.0 \
    torch-ema==0.3 \
    nvidia-cuda-nvcc-cu12==12.8.93 \
    lpips==0.1.4 \
    piq==0.8.0 \
    lightning==2.3.3 \
    pytorch-lightning==2.3.3 \
    huggingface-hub==0.24.5 \
    transformers==4.46.3 \
    tokenizers==0.20.3

# ------------------------------------------------------------
# 3. Install dctorch
#
# dctorch declares NumPy < 2 in its old package metadata.
# Python 3.13 uses NumPy 2 here, so dependencies are deliberately
# not resolved from dctorch's outdated metadata.
# ------------------------------------------------------------

%pip install -q --no-deps dctorch==0.1.2

# ------------------------------------------------------------
# 4. Install the NATTEN wheel that EXACTLY matches:
#       torch 2.11.0
#       CUDA 12.8
# ------------------------------------------------------------

%pip install -q --no-cache-dir \
    "natten==0.21.6+torch2110cu128" \
    -f https://whl.natten.org

# ------------------------------------------------------------
# 5. Install the exact torch-fidelity revision seen in your
#    original notebook, rather than an unpinned future revision.
# ------------------------------------------------------------

%pip install -q \
    "git+https://github.com/toshas/torch-fidelity.git@5e211a950a7b45206bd4976813ffd6aed6cf4ccc"


# ------------------------------------------------------------
# 6. Install BasicSR (required only for degradation='difface')
#
# basicsr 1.4.2's setup.py reads its version via locals() after
# exec(). Python 3.13 (PEP 667) makes locals() a snapshot, so that
# lookup raises KeyError. The source is patched before installing.
# ------------------------------------------------------------

%pip install -q cython

!rm -rf /tmp/basicsr_build && mkdir -p /tmp/basicsr_build
!cd /tmp/basicsr_build && curl -sL -o basicsr.tar.gz \
    https://files.pythonhosted.org/packages/source/b/basicsr/basicsr-1.4.2.tar.gz && \
    tar xzf basicsr.tar.gz

import pathlib

setup_py = pathlib.Path("/tmp/basicsr_build/basicsr-1.4.2/setup.py")
source = setup_py.read_text()

old_get_version = """def get_version():
    with open(version_file, 'r') as f:
        exec(compile(f.read(), version_file, 'exec'))
    return locals()['__version__']"""

new_get_version = """def get_version():
    ns = {}
    with open(version_file, 'r') as f:
        exec(compile(f.read(), version_file, 'exec'), ns)
    return ns['__version__']"""

if old_get_version in source:
    setup_py.write_text(source.replace(old_get_version, new_get_version, 1))
    print("Patched basicsr setup.py for Python 3.13")
elif new_get_version in source:
    print("basicsr setup.py already patched")
else:
    raise RuntimeError("basicsr setup.py get_version() differs from expected source")

%pip install -q --use-pep517 --no-build-isolation /tmp/basicsr_build/basicsr-1.4.2


# ------------------------------------------------------------
# 7. Patch BasicSR for torchvision 0.26
#
# basicsr 1.4.2 imports torchvision.transforms.functional_tensor,
# which was removed in modern torchvision.
# ------------------------------------------------------------

!FILE=$(python -c "import importlib.metadata, pathlib; print(pathlib.Path(importlib.metadata.distribution('basicsr').locate_file('basicsr/data/degradations.py')))") && \
echo "Patching $FILE ..." && \
sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' "$FILE" && \
grep -n "rgb_to_grayscale" "$FILE" | head

# ------------------------------------------------------------
# Final pre-restart sanity check in a fresh subprocess
# ------------------------------------------------------------

!python -c "import torch, torchvision, torchaudio; \
print('torch:', torch.__version__); \
print('torchvision:', torchvision.__version__); \
print('torchaudio:', torchaudio.__version__); \
print('torch CUDA:', torch.version.cuda); \
assert torch.__version__ == '2.11.0+cu128'; \
assert torchvision.__version__ == '0.26.0+cu128'; \
assert torchaudio.__version__ == '2.11.0+cu128'; \
assert torch.version.cuda == '12.8'"

!python -c "from importlib.metadata import version; \
print('natten:', version('natten')); \
assert version('natten') == '0.21.6+torch2110cu128'"

print()
print("Installation completed.")
print("Now restart the runtime before running any imports.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
scipy 1.15.3 requires numpy<2.5,>=1.23.5, but you have numpy 2.5.2 which is incompatible.
lightning 2.3.3 requires fsspec[http]<2026.0,>=2022.5.0, but you have fsspec 2026.7.0 which is incompatible.
dctorch 0.1.2 requires numpy<2.0.0,>=1.22.3, but you have numpy 2.5.2 which is incompatible.
libcuvs-cu13 26.6.0 requires cuda-toolkit[cublas,curand,cusolver,cusparse,nvrtc]==13.*, but you have cuda-toolkit 12.8.1 which is incompatible.
gradio 6.27.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.24.5 which is incompatible.
libcuml-cu13 26.6.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==13.*, but you have cuda-toolkit 12.8.1 which is incompatible.
peft 0.21.0 requires huggingface_hub>=0.25.0, but you have huggingface-hu

## Restart the runtime
After the installation cell finishes, use:

Runtime → Restart session

In [ ]:
# ============================================================
# RUN THIS CELL AFTER RESTARTING THE RUNTIME
# ============================================================

import sys
from importlib.metadata import version

import cv2
import dctorch
import einops
import lightning
import lovely_tensors as lt
import lpips
import natten
import numpy as np
import piq
import pytorch_lightning
import scipy
import timm
import tokenizers
import torch
import torch_ema
import torch_fidelity
import torchaudio
import torchvision
import transformers
import wandb
import basicsr


# ------------------------------------------------------------
# Display all important package versions
# ------------------------------------------------------------

packages = [
    "numpy",
    "scipy",
    "torch",
    "torchvision",
    "torchaudio",
    "opencv-python",
    "timm",
    "wandb",
    "lovely-tensors",
    "einops",
    "torch-ema",
    "nvidia-cuda-nvcc-cu12",
    "lpips",
    "piq",
    "lightning",
    "pytorch-lightning",
    "huggingface-hub",
    "transformers",
    "tokenizers",
    "dctorch",
    "natten",
    "torch-fidelity",
    "basicsr"
]

print("Installed environment")
print("=" * 55)
print(f"{'Python':28s} {sys.version.split()[0]}")

for package in packages:
    print(f"{package:28s} {version(package)}")


# ------------------------------------------------------------
# Verify the versions that must match exactly
# ------------------------------------------------------------

assert sys.version_info[:2] == (3, 13), (
    f"Expected Python 3.13, found {sys.version.split()[0]}"
)

assert torch.__version__ == "2.11.0+cu128", (
    f"Expected Torch 2.11.0+cu128, found {torch.__version__}"
)

assert torchvision.__version__ == "0.26.0+cu128", (
    f"Expected Torchvision 0.26.0+cu128, found {torchvision.__version__}"
)

assert torchaudio.__version__ == "2.11.0+cu128", (
    f"Expected Torchaudio 2.11.0+cu128, found {torchaudio.__version__}"
)

assert torch.version.cuda == "12.8", (
    f"Expected CUDA 12.8 PyTorch build, found {torch.version.cuda}"
)

assert transformers.__version__ == "4.46.3", (
    f"Unexpected Transformers version: {transformers.__version__}"
)

assert tokenizers.__version__ == "0.20.3", (
    f"Unexpected Tokenizers version: {tokenizers.__version__}"
)

assert version("natten") == "0.21.6+torch2110cu128", (
    f"Unexpected NATTEN version: {version('natten')}"
)

assert version("basicsr") == "1.4.2", (
    f"Unexpected BasicSR version: {version('basicsr')}"
)

# ------------------------------------------------------------
# Check the GPU and compiled NATTEN extension
# ------------------------------------------------------------

assert torch.cuda.is_available(), (
    "CUDA is unavailable. Change the Colab runtime type to GPU."
)

assert natten.HAS_LIBNATTEN, (
    "NATTEN imported, but its compiled CUDA library is unavailable."
)

print()
print("GPU information")
print("=" * 55)
print("CUDA available:       ", torch.cuda.is_available())
print("CUDA runtime:         ", torch.version.cuda)
print("GPU:                  ", torch.cuda.get_device_name(0))
print("Compute capability:   ", torch.cuda.get_device_capability(0))
print("NATTEN CUDA library:  ", natten.HAS_LIBNATTEN)


# ------------------------------------------------------------
# Basic CUDA calculation
# ------------------------------------------------------------

x = torch.randn(256, 256, device="cuda")
y = x @ x
torch.cuda.synchronize()

assert torch.isfinite(y).all()
print("CUDA tensor test:      OK")


# ------------------------------------------------------------
# Test dctorch with the installed NumPy/SciPy versions
# ------------------------------------------------------------

from dctorch.functional import dct2

dct_input = torch.randn(1, 8, 8, device="cuda")
dct_output = dct2(dct_input)

assert dct_output.shape == dct_input.shape
assert torch.isfinite(dct_output).all()

print("dctorch test:          OK")
print()
print("Environment is ready.")

Installed environment
Python                       3.13.15
numpy                        2.1.3
scipy                        1.15.3
torch                        2.11.0+cu128
torchvision                  0.26.0+cu128
torchaudio                   2.11.0+cu128
opencv-python                4.10.0.84
timm                         1.0.8
wandb                        0.30.0
lovely-tensors               0.1.16
einops                       0.8.0
torch-ema                    0.3
nvidia-cuda-nvcc-cu12        12.8.93
lpips                        0.1.4
piq                          0.8.0
lightning                    2.3.3
pytorch-lightning            2.3.3
huggingface-hub              0.24.5
transformers                 4.46.3
tokenizers                   0.20.3
dctorch                      0.1.2
natten                       0.21.6+torch2110cu128
torch-fidelity               0.4.0
basicsr                      1.4.2

GPU information
CUDA available:        True
CUDA runtime:          12.8
GPU:            

## Connecting Drive to Colab

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# ============================================================
# FINISH PMRF COMPATIBILITY SETUP
#
# ============================================================

from pathlib import Path
import os
import shutil

PMRF_ROOT = Path("/content/drive/MyDrive/PMRF")

NATTEN_FILE = (
    PMRF_ROOT
    / "arch"
    / "hourglass"
    / "image_transformer_v2.py"
)

DEGRADATION_FILE = (
    PMRF_ROOT
    / "utils"
    / "create_degradation.py"
)


# ------------------------------------------------------------
# 1. Verify the already-present NATTEN compatibility patch
# ------------------------------------------------------------

if not NATTEN_FILE.is_file():
    raise FileNotFoundError(
        f"NATTEN source file was not found:\n{NATTEN_FILE}"
    )

natten_text = NATTEN_FILE.read_text(encoding="utf-8")

existing_natten_guard = (
    'getattr(natten, "has_fused_na", lambda: True)()'
)

assert existing_natten_guard in natten_text, (
    "The expected existing NATTEN compatibility guard was not found."
)

assert "natten.functional.na2d(" in natten_text, (
    "The modern NATTEN na2d call was not found."
)

print("NATTEN compatibility patch: already present and valid")


# ------------------------------------------------------------
# 2. Make BasicSR optional
#
# BasicSR is only needed for degradation='difface'.
# The other PMRF degradations can work without it.
# ------------------------------------------------------------

if not DEGRADATION_FILE.is_file():
    raise FileNotFoundError(
        f"Degradation source file was not found:\n{DEGRADATION_FILE}"
    )

text = DEGRADATION_FILE.read_text(encoding="utf-8")

marker = "_BASICSR_IMPORT_ERROR"

old_import_block = """from basicsr.data import degradations as degradations
from basicsr.data.transforms import augment
from basicsr.utils import img2tensor"""

new_import_block = """try:
    from basicsr.data import degradations as degradations
    from basicsr.data.transforms import augment
    from basicsr.utils import img2tensor
except (ModuleNotFoundError, ImportError) as exc:
    _BASICSR_IMPORT_ERROR = exc

    class _MissingBasicSR:
        def __getattr__(self, name):
            raise ModuleNotFoundError(
                "BasicSR is required for degradation='difface', "
                "but it is not installed in this environment."
            ) from _BASICSR_IMPORT_ERROR

    def _missing_basicsr(*args, **kwargs):
        raise ModuleNotFoundError(
            "BasicSR is required for degradation='difface', "
            "but it is not installed in this environment."
        ) from _BASICSR_IMPORT_ERROR

    degradations = _MissingBasicSR()
    augment = _missing_basicsr
    img2tensor = _missing_basicsr"""


if marker in text:
    print("Optional BasicSR patch: already present")

elif old_import_block in text:
    backup = DEGRADATION_FILE.with_name(
        DEGRADATION_FILE.name + ".before_optional_basicsr.bak"
    )

    if not backup.exists():
        shutil.copy2(DEGRADATION_FILE, backup)
        print(f"Created backup: {backup}")

    patched_text = text.replace(
        old_import_block,
        new_import_block,
        1,
    )

    DEGRADATION_FILE.write_text(
        patched_text,
        encoding="utf-8",
    )

    verification_text = DEGRADATION_FILE.read_text(
        encoding="utf-8"
    )

    assert marker in verification_text, (
        "BasicSR patch verification failed."
    )

    print("Optional BasicSR patch: applied successfully")

else:
    print("\nRelevant BasicSR lines:")

    for number, line in enumerate(
        text.splitlines(),
        start=1,
    ):
        if "basicsr" in line.lower():
            print(f"{number:4d}: {line}")

    raise RuntimeError(
        "The BasicSR imports differ from the expected source. "
        "No BasicSR modification was made."
    )


# ------------------------------------------------------------
# 3. Restore older torch.load behavior for your own trusted
#    PMRF / Lightning checkpoints
# ------------------------------------------------------------

os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

print()
print("=" * 70)
print("PMRF compatibility setup completed.")
print("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1")
print("=" * 70)

NATTEN compatibility patch: already present and valid
Optional BasicSR patch: already present

PMRF compatibility setup completed.
TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1


In [ ]:
%%bash
cd /content/drive/MyDrive/PMRF
python - <<'PY'
import inspect
import os

os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

import natten
import torch

print("Torch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("NATTEN:", natten.__version__)
print("GPU:", torch.cuda.get_device_name(0))

assert torch.cuda.is_available(), "CUDA is unavailable."

assert natten.HAS_LIBNATTEN, (
    "NATTEN imported, but its compiled CUDA library is unavailable."
)


# ------------------------------------------------------------
# Import the real PMRF project modules
# ------------------------------------------------------------

from arch.hourglass.image_transformer_v2 import (
    NeighborhoodSelfAttentionBlock,
)
from lightning_models.mmse_rectified_flow import (
    MMSERectifiedFlow,
)
from utils.create_degradation import create_degradation

print("PMRF module imports: OK")


# ------------------------------------------------------------
# Confirm the source contains your existing valid NATTEN guard
# ------------------------------------------------------------

forward_source = inspect.getsource(
    NeighborhoodSelfAttentionBlock.forward
)

expected_guard = (
    'getattr(natten, "has_fused_na", lambda: True)()'
)

assert expected_guard in forward_source, (
    "The expected NATTEN compatibility guard was not found."
)

assert "natten.functional.na2d(" in forward_source, (
    "The modern NATTEN na2d branch was not found."
)

print("PMRF NATTEN source compatibility: OK")


# ------------------------------------------------------------
# Verify all non-DifFace degradation constructors
# ------------------------------------------------------------

degradation_names = [
    "sr_bicubic_x8_gaussian_noise_005",
    "gaussian_noise_035",
    "colorization_gaussian_noise_025",
    "random_inpainting_gaussian_noise_01",
]

for name in degradation_names:
    degradation = create_degradation(name)

    assert callable(degradation), (
        f"Degradation {name!r} is not callable."
    )

    print(f"Degradation {name}: OK")


# ------------------------------------------------------------
# Execute the ACTUAL PMRF NeighborhoodSelfAttentionBlock
# ------------------------------------------------------------

device = torch.device("cuda")
dtype = torch.float16

block = NeighborhoodSelfAttentionBlock(
    d_model=64,
    d_head=32,
    cond_features=64,
    kernel_size=3,
    dropout=0.0,
).to(device=device, dtype=dtype)

batch_size = 1
height = 8
width = 8

x = torch.randn(
    batch_size,
    height,
    width,
    64,
    device=device,
    dtype=dtype,
    requires_grad=True,
)

cond = torch.randn(
    batch_size,
    64,
    device=device,
    dtype=dtype,
)

row_positions = torch.arange(
    height,
    device=device,
    dtype=dtype,
)

column_positions = torch.arange(
    width,
    device=device,
    dtype=dtype,
)

grid_y, grid_x = torch.meshgrid(
    row_positions,
    column_positions,
    indexing="ij",
)

pos = torch.stack(
    (grid_y, grid_x),
    dim=-1,
)

output = block(x, pos, cond)

assert output.shape == x.shape, (
    f"Unexpected output shape: {output.shape}"
)

assert torch.isfinite(output).all(), (
    "PMRF attention output contains NaN or infinity."
)

loss = output.float().square().mean()
loss.backward()
torch.cuda.synchronize()

assert x.grad is not None, (
    "No gradient was produced for the PMRF attention input."
)

assert torch.isfinite(x.grad).all(), (
    "PMRF attention gradient contains NaN or infinity."
)

print("PMRF NeighborhoodSelfAttentionBlock forward/backward: OK")
print()
print("PMRF compatibility smoke test passed.")
PY

Torch: 2.11.0+cu128
CUDA runtime: 12.8
NATTEN: 0.21.6
GPU: Tesla T4
PMRF module imports: OK
PMRF NATTEN source compatibility: OK
Degradation sr_bicubic_x8_gaussian_noise_005: OK
Degradation gaussian_noise_035: OK
Degradation colorization_gaussian_noise_025: OK
Degradation random_inpainting_gaussian_noise_01: OK
PMRF NeighborhoodSelfAttentionBlock forward/backward: OK

PMRF compatibility smoke test passed.


# (Resizing and Cropping Data)

In [ ]:
import os
from PIL import Image, ImageOps
from tqdm import tqdm

# === CONFIG ===
ROOT = "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test"
TARGET_SIZE = 128

# === FUNCTION ===
def resize_center_crop(img, size=128):
    img = ImageOps.exif_transpose(img).convert("RGB")
    w, h = img.size
    scale = size / min(w, h)
    new_w, new_h = int(round(w * scale)), int(round(h * scale))
    img = img.resize((new_w, new_h), Image.LANCZOS)
    left, top = (new_w - size) // 2, (new_h - size) // 2
    img = img.crop((left, top, left + size, top + size))
    return img

# === MAIN LOOP ===
# Collect all image paths (recursively)
all_images = []
for root, dirs, files in os.walk(ROOT):
    for fname in files:
        if fname.lower().endswith((".jpg", ".jpeg", ".png", ".JPEG")):
            all_images.append(os.path.join(root, fname))

print(f"Found {len(all_images)} images. Resizing and flattening...")

# Create a temporary folder to avoid overwriting while processing
TMP_DIR = os.path.join(ROOT, "_tmp_resized")
os.makedirs(TMP_DIR, exist_ok=True)

for src_path in tqdm(all_images, desc="Resizing images"):
    fname = os.path.basename(src_path)
    dst_path = os.path.join(TMP_DIR, fname)
    try:
        with Image.open(src_path) as img:
            out = resize_center_crop(img, TARGET_SIZE)
            out.save(dst_path, "JPEG", quality=95)
    except Exception as e:
        print(f"❌ Error on {src_path}: {e}")

# Remove old folders and move all resized files into the main test folder
print("🧹 Cleaning up old subfolders and moving resized images...")
for item in os.listdir(ROOT):
    item_path = os.path.join(ROOT, item)
    if os.path.isdir(item_path) and item != "_tmp_resized":
        for f in os.listdir(item_path):
            try:
                os.remove(os.path.join(item_path, f))
            except:
                pass
        os.rmdir(item_path)

# Move everything from _tmp_resized back to ROOT
for f in os.listdir(TMP_DIR):
    os.rename(os.path.join(TMP_DIR, f), os.path.join(ROOT, f))

os.rmdir(TMP_DIR)

print(f"✅ All {len(all_images)} images flattened and resized to {TARGET_SIZE}×{TARGET_SIZE}")
print(f"📁 Saved directly in: {ROOT}")


Found 5000 images. Resizing and flattening...


Resizing images: 100%|██████████| 5000/5000 [03:21<00:00, 24.87it/s]


🧹 Cleaning up old subfolders and moving resized images...
✅ All 5000 images flattened and resized to 128×128
📁 Saved directly in: /content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test


# 🧠 ========== IMAGENET DATA ========== 🧠

# Training 💻 ------------------------- 💻

## Colorization

### Stage 1

In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"   # <--- must be set before lightning imports
import matplotlib
matplotlib.use("Agg")


In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_mmse.sh


/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.2 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251009_095416-2zfw6am1
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run crimson-sun-3
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/2zfw6am1
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
---------------------------------------------------------------------------------------

### Stage 2

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_pmrf.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.2 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251009_133754-elebleld
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run wobbly-field-4
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/elebleld
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/t

### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_naive_flow.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.2 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251012_094428-j8v0ow0f
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run eager-morning-16
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/j8v0ow0f
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted 

### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_y.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.2 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251013_012315-z4kmgcpy
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run volcanic-salad-20
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/z4kmgcpy
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted

### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_mmse_model.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.2 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251013_140357-kv489mn8
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run vivid-microwave-24
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/kv489mn8
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracte

## Gaussian Noise

### Stage 1

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_mmse.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.2 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251010_134110-36przncv
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run lunar-lake-7
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/36przncv
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------

### Stage 2

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_pmrf.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.2 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251010_190316-0d150a4a
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run vivid-firebrand-8
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/0d150a4a
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted

### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_naive_flow.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.2 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251012_125208-7pqtnsnr
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run dainty-violet-17
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/7pqtnsnr
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted 

### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_y.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.2 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251013_043724-i2s1xfpo
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run still-feather-21
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/i2s1xfpo
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted 

### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_mmse_model.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.2 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251013_232931-srm2wd54
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run bumbling-butterfly-28
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/srm2wd54
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extra

## Super-resolution

### Stage 1

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_mmse.sh

/content/drive/MyDrive/PMRF
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: 2
wandb: You chose 'Use an existing W&B account'
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: wandb version 0.22.2 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251011_074402-1w3muqq4
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run toasty-sun-11
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/dani

### Stage 2

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_pmrf.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.2 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251011_130103-hu1ugkpl
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run noble-lake-12
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/hu1ugkpl
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/to

### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_naive_flow.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.2 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251012_161121-rh1jbixe
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run floral-butterfly-18
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/rh1jbixe
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extract

### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_y.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.2 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251013_074640-sel8gutl
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run driven-gorge-22
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/sel8gutl
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted f

### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_mmse_model.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.2 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251014_034116-42nd5f79
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run dazzling-salad-29
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/42nd5f79
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted

## Inpainting

### Stage 1

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_mmse.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.2 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251011_224628-zt6z1jm3
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run distinctive-dawn-14
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/zt6z1jm3
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
---------------------------------------------------------------------------------

### Stage 2

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_pmrf.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.2 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251012_033610-othpxpy3
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run different-lake-15
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/othpxpy3
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cach

### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_naive_flow.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.2 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251012_230244-cgpd5ssa
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run inpainting_naive_flow
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/cgpd5ssa
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/

### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_y.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.2 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251023_180021-o3xrucm0
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run zesty-dew-32
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/o3xrucm0
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/tor

### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_mmse_model.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.2 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251014_075335-rbvl2asz
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run clear-music-30
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/rbvl2asz
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted fe

# Inference ⚡ ------------------------- ⚡

In [ ]:
PMRF_CKPT = "/content/drive/MyDrive/PMRF/PMRF/inpainting_pmrf/checkpoints/last.ckpt"   # stage-2
MMSE_CKPT = "/content/drive/MyDrive/PMRF/PMRF/inpainting_mmse/checkpoints/last.ckpt"   # stage-1
NF_CKPT="/content/drive/MyDrive/PMRF/PMRF/inpainting_naive_flow/checkpoints/last.ckpt"
PCY_CKPT="/content/drive/MyDrive/PMRF/PMRF/inpainting_post_con_on_y/checkpoints/last.ckpt"
PCMMSE_CKPT="/content/drive/MyDrive/PMRF/PMRF/inpainting_post_con_on_mmse/checkpoints/last.ckpt"

TEST_ROOT = "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test"  # <-- your flat 5k images

OUT_PMRF  = "/content/drive/MyDrive/PMRF/outputs/imagenet_data/inpainting_pmrf_test_K100"
OUT_MMSE  = "/content/drive/MyDrive/PMRF/outputs/imagenet_data/inpainting_mmse_test_K100"
OUT_NF="/content/drive/MyDrive/PMRF/outputs/imagenet_data/inpainting_naive_flow_test_K100"
OUT_PCY="/content/drive/MyDrive/PMRF/outputs/imagenet_data/inpainting_post_con_on_y_test_K100"
OUT_PCMMSE="/content/drive/MyDrive/PMRF/outputs/imagenet_data/inpainting_post_con_on_mmse_test_K100"

## Colorization

### PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PMRF"

!python test.py \
  --precision "32" \
  --degradation "colorization_gaussian_noise_025" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PMRF_CKPT" \
  --results_path "$OUT_PMRF" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
/usr/local/lib/python3.11/site-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 5 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-incepti

### MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_MMSE"

!python test.py \
  --precision "32" \
  --degradation "colorization_gaussian_noise_025" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$MMSE_CKPT" \
  --results_path "$OUT_MMSE" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Testing DataLoader 0: 100% 79/79 [02:06<00:00,  1.60s/it]


### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_NF"

!python test.py \
  --precision "32" \
  --degradation "colorization_gaussian_noise_025" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$NF_CKPT" \
  --results_path "$OUT_NF" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Testing Data

### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCY"

!python test.py \
  --precision "32" \
  --degradation "colorization_gaussian_noise_025" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCY_CKPT" \
  --results_path "$OUT_PCY" \
  --num_flow_steps 100

/content/drive/MyDrive/PMRF
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Testing Data

### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCMMSE"

!python test.py \
  --precision "32" \
  --degradation "colorization_gaussian_noise_025" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCMMSE_CKPT" \
  --results_path "$OUT_PCMMSE" \
  --num_flow_steps 100

/content/drive/MyDrive/PMRF
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Testing Data

## Gaussian Noise

### PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PMRF"

!python test.py \
  --precision "32" \
  --degradation "gaussian_noise_035" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PMRF_CKPT" \
  --results_path "$OUT_PMRF" \
  --num_flow_steps 100

/content/drive/MyDrive/PMRF
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Testing Data

### MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_MMSE"

!python test.py \
  --precision "32" \
  --degradation "gaussian_noise_035" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$MMSE_CKPT" \
  --results_path "$OUT_MMSE" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Testing DataLoader 0: 100% 79/79 [02:42<00:00,  2.05s/it]


### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_NF"

!python test.py \
  --precision "32" \
  --degradation "gaussian_noise_035" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$NF_CKPT" \
  --results_path "$OUT_NF" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Testing Data

### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCY"

!python test.py \
  --precision "32" \
  --degradation "gaussian_noise_035" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCY_CKPT" \
  --results_path "$OUT_PCY" \
  --num_flow_steps 100

/content/drive/MyDrive/PMRF
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Testing Data

### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCMMSE"

!python test.py \
  --precision "32" \
  --degradation "gaussian_noise_035" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCMMSE_CKPT" \
  --results_path "$OUT_PCMMSE" \
  --num_flow_steps 100

/content/drive/MyDrive/PMRF
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Testing Data

## Super-resolution

### PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PMRF"

!python test.py \
  --precision "32" \
  --degradation "sr_bicubic_x8_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PMRF_CKPT" \
  --results_path "$OUT_PMRF" \
  --num_flow_steps 100

/content/drive/MyDrive/PMRF
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Testing Data

### MMSE

In [ ]:
MMSE_CKPT = "/content/drive/MyDrive/PMRF/PMRF/super_resolution_mmse/checkpoints/last.ckpt"
TEST_ROOT = "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test"
OUT_MMSE  = "/content/drive/MyDrive/PMRF/outputs/imagenet_data/super_resolution_mmse_test_K100"

%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_MMSE"

!python test.py \
  --precision "32" \
  --degradation "sr_bicubic_x8_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$MMSE_CKPT" \
  --results_path "$OUT_MMSE" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Testing DataLoader 0: 100% 79/79 [1:22:38<00:00, 62.76s/it]


### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_NF"

!python test.py \
  --precision "32" \
  --degradation "sr_bicubic_x8_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$NF_CKPT" \
  --results_path "$OUT_NF" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Testing Data

### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCY"

!python test.py \
  --precision "32" \
  --degradation "sr_bicubic_x8_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCY_CKPT" \
  --results_path "$OUT_PCY" \
  --num_flow_steps 100

/content/drive/MyDrive/PMRF
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Testing Data

### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCMMSE"

!python test.py \
  --precision "32" \
  --degradation "sr_bicubic_x8_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCMMSE_CKPT" \
  --results_path "$OUT_PCMMSE" \
  --num_flow_steps 100

/content/drive/MyDrive/PMRF
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Testing Data

## Inpainting

### PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PMRF"

!python test.py \
  --precision "32" \
  --degradation "random_inpainting_gaussian_noise_01" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PMRF_CKPT" \
  --results_path "$OUT_PMRF" \
  --num_flow_steps 100

/content/drive/MyDrive/PMRF
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Testing Data

### MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PMRF"

!python test.py \
  --precision "32" \
  --degradation "random_inpainting_gaussian_noise_01" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$MMSE_CKPT" \
  --results_path "$OUT_MMSE" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Testing DataLoader 0: 100% 79/79 [02:23<00:00,  1.81s/it]


### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_NF"

!python test.py \
  --precision "32" \
  --degradation "random_inpainting_gaussian_noise_01" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$NF_CKPT" \
  --results_path "$OUT_NF" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Testing Data

### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCY"

!python test.py \
  --precision "32" \
  --degradation "random_inpainting_gaussian_noise_01" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCY_CKPT" \
  --results_path "$OUT_PCY" \
  --num_flow_steps 100

/content/drive/MyDrive/PMRF
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100% 91.2M/91.2M [00:04<00:00, 19.5MB/s]
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
d

### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCMMSE"

!python test.py \
  --precision "32" \
  --degradation "random_inpainting_gaussian_noise_01" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCMMSE_CKPT" \
  --results_path "$OUT_PCMMSE" \
  --num_flow_steps 100

/content/drive/MyDrive/PMRF
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Testing Data

# Evaluation 🔎 ------------------------- 🔍

## Colorization

### PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/colorization_pmrf_test_K100/colorization_gaussian_noise_025/pmrf/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [05:16<00:00,  7.91s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

### MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/colorization_mmse_test_K100/colorization_gaussian_noise_025/mmse/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [05:17<00:00,  7.93s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/colorization_naive_flow_test_K100/colorization_gaussian_noise_025/naive_flow/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:54<00:00,  1.36s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/colorization_post_con_on_y_test_K100/colorization_gaussian_noise_025/posterior_conditioned_on_y/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:13<00:00,  2.90it/s]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/colorization_post_con_on_mmse_test_K100/colorization_gaussian_noise_025/posterior_conditioned_on_mmse/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:13<00:00,  2.92it/s]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

## Gaussian Noise

### PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/gaussian_noise_pmrf_test_K100/gaussian_noise_035/pmrf/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [04:35<00:00,  6.88s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

### MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/gaussian_noise_mmse_test_K100/gaussian_noise_035/mmse/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [05:13<00:00,  7.84s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/gaussian_noise_naive_flow_test_K100/gaussian_noise_035/naive_flow/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:51<00:00,  1.28s/it]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/gaussian_noise_post_con_on_y_test_K100/gaussian_noise_035/posterior_conditioned_on_y/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:13<00:00,  2.92it/s]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/gaussian_noise_post_con_on_mmse_test_K100/gaussian_noise_035/posterior_conditioned_on_mmse/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:13<00:00,  2.88it/s]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

## Super-resolution

### PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/super_resolution_pmrf_test_K100/sr_bicubic_x8_gaussian_noise_005/pmrf/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [04:27<00:00,  6.68s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

### MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/super_resolution_mmse_test_K100/sr_bicubic_x8_gaussian_noise_005/mmse/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:49<00:00,  1.24s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/super_resolution_naive_flow_test_K100/sr_bicubic_x8_gaussian_noise_005/naive_flow/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:13<00:00,  2.93it/s]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/super_resolution_post_con_on_y_test_K100/sr_bicubic_x8_gaussian_noise_005/posterior_conditioned_on_y/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:13<00:00,  2.87it/s]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/super_resolution_post_con_on_mmse_test_K100/sr_bicubic_x8_gaussian_noise_005/posterior_conditioned_on_mmse/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:13<00:00,  2.90it/s]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

## Inpainting

### PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/inpainting_pmrf_test_K100/random_inpainting_gaussian_noise_01/pmrf/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:50<00:00,  1.26s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

### MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/inpainting_mmse_test_K100/random_inpainting_gaussian_noise_01/mmse/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [05:03<00:00,  7.58s/it]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/inpainting_naive_flow_test_K100/random_inpainting_gaussian_noise_01/naive_flow/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:47<00:00,  1.19s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/inpainting_post_con_on_y_test_K100/random_inpainting_gaussian_noise_01/posterior_conditioned_on_y/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
Downloading: "https://github.com/photosynthesis-team/photosynthesis.metrics/releases/download/v0.4.0/lpips_weights.pt" to /root/.cache/torch/hub/checkpoints/lpips_weights.pt
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100% 40/40 [00:49<00:00,  1.23s/it]
Creatin

### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/inpainting_post_con_on_mmse_test_K100/random_inpainting_gaussian_noise_01/posterior_conditioned_on_mmse/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:13<00:00,  3.05it/s]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

# 🧠 ========== ZAPPOS DATA ========== 🧠

# Training

## Colorization

### Stage 1

In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"   # <--- must be set before lightning imports
import matplotlib
matplotlib.use("Agg")


In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_mmse.sh


/content/drive/MyDrive/PMRF
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: 2
wandb: You chose 'Use an existing W&B account'
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: wandb version 0.22.2 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251028_231536-8dnacb5v
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run still-dust-33
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/dani

### Stage 2

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_pmrf.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.3 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251029_170248-46d4kfa4
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run wobbly-monkey-37
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/46d4kfa4
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted 

### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_naive_flow.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.3 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251029_051021-9gncnmow
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run comfy-snowflake-34
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/9gncnmow
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cac

### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_y.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.3 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251029_084734-xcan1yi5
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run brisk-voice-35
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/xcan1yi5
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted fe

### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_mmse_model.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.3 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251029_231709-lflz7gc6
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run smooth-water-40
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/lflz7gc6
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/

## Gaussian Noise

### Stage 1

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_mmse.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.3 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251030_051332-1j30ct6z
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run muted-bat-41
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/1j30ct6z
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------

### Stage 2

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_pmrf.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.3 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251030_121208-g4k8ngu3
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run occult-mummy-43
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/g4k8ngu3
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted f

### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_naive_flow.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.3 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251030_083117-brvm3k84
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run faded-whisper-42
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/brvm3k84
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted 

### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_y.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.3 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251030_164551-vu56zglg
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run scary-imp-44
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/vu56zglg
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted feat

### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_mmse_model.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.3 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251031_074141-sba2jlyq
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run tenebrous-broomstick-47
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/sba2jlyq
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root

## Super-resolution

### Stage 1

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_mmse.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.3 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251031_142304-dbna3keg
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run nocturnal-seance-48
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/dbna3keg
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
---------------------------------------------------------------------------------

### Stage 2

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_pmrf.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.3 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251101_165426-cmo0ojdy
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run effortless-hill-52
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/cmo0ojdy
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cac

### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_naive_flow.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.3 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251031_174034-7dfc0qsn
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run strange-poltergeist-49
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/7dfc0qsn
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extr

### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_y.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: 2
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Wai

### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_mmse_model.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.3 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251102_101649-lumds7o2
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run still-dawn-55
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/lumds7o2
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/to

## Inpainting

### Stage 1

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_mmse.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run 16v59jam (0.5s)
wandb: ⣯ setting up run 16v59jam (0.5s)
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in wandb/run-20260510_065827-16v59jam
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run stellar-deluge-62
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-l

### Stage 2

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_pmrf.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in wandb/run-20260510_114654-9cyvsi0q
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run radiant-brook-67
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/9cyvsi0q
wandb: Detected [huggingface_hub.inference] in use.
wandb:

### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_naive_flow.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.3 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251102_185455-yhetzjww
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run fiery-darkness-58
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/yhetzjww
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted

### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_y.sh

/content/drive/MyDrive/PMRF
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.22.3 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in ./wandb/run-20251102_232932-jcxc3orl
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run comic-frog-59
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/jcxc3orl
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted fea

### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_mmse_model.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run sz9oqo02 (0.5s)
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in wandb/run-20260510_192114-sz9oqo02
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run astral-frost-71
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/sz9oqo02
wandb: Detected [hu

# Inference

In [ ]:
PMRF_CKPT = "/content/drive/MyDrive/PMRF/PMRF/zappos_data/zap_inpainting_pmrf/checkpoints/last.ckpt"   # stage-2
MMSE_CKPT = "/content/drive/MyDrive/PMRF/PMRF/zappos_data/zap_inpainting_mmse/checkpoints/last.ckpt"   # stage-1
NF_CKPT="/content/drive/MyDrive/PMRF/PMRF/zappos_data/zap_inpainting_naive_flow/checkpoints/last.ckpt"
PCY_CKPT="/content/drive/MyDrive/PMRF/PMRF/zappos_data/zap_sr_post_con_on_y/checkpoints/last.ckpt"
PCMMSE_CKPT="/content/drive/MyDrive/PMRF/PMRF/zappos_data/zap_inpainting_post_con_on_mmse/checkpoints/last.ckpt"

TEST_ROOT = "/content/drive/MyDrive/PMRF/data/zap50k_128/test"  # <-- your flat 5k images

OUT_PMRF  = "/content/drive/MyDrive/PMRF/outputs/zappos_data/inpainting_pmrf_test_K100"
OUT_MMSE  = "/content/drive/MyDrive/PMRF/outputs/zappos_data/inpainting_mmse_test_K100"
OUT_NF="/content/drive/MyDrive/PMRF/outputs/zappos_data/inpainting_naive_flow_test_K100"
OUT_PCY="/content/drive/MyDrive/PMRF/outputs/zappos_data/sr_post_con_on_y_test_K100"
OUT_PCMMSE="/content/drive/MyDrive/PMRF/outputs/zappos_data/inpainting_post_con_on_mmse_test_K100"

## Colorization

### PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PMRF"

!python test.py \
  --precision "32" \
  --degradation "colorization_gaussian_noise_025" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PMRF_CKPT" \
  --results_path "$OUT_PMRF" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100% 91.2M/91.2M [00:01<00:00, 66.6MB/s]
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatical

### MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_MMSE"

!python test.py \
  --precision "32" \
  --degradation "colorization_gaussian_noise_025" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$MMSE_CKPT" \
  --results_path "$OUT_MMSE" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

2026-05-22 16:20:36.086717: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see sl

### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_NF"

!python test.py \
  --precision "32" \
  --degradation "colorization_gaussian_noise_025" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$NF_CKPT" \
  --results_path "$OUT_NF" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCY"

!python test.py \
  --precision "32" \
  --degradation "colorization_gaussian_noise_025" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCY_CKPT" \
  --results_path "$OUT_PCY" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCMMSE"

!python test.py \
  --precision "32" \
  --degradation "colorization_gaussian_noise_025" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCMMSE_CKPT" \
  --results_path "$OUT_PCMMSE" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

## Gaussian Noise

### PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PMRF"

!python test.py \
  --precision "32" \
  --degradation "gaussian_noise_035" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PMRF_CKPT" \
  --results_path "$OUT_PMRF" \
  --num_flow_steps 100

/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100% 91.2M/91.2M [00:01<00:00, 67.1MB/s]
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatical

### MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_MMSE"

!python test.py \
  --precision "32" \
  --degradation "gaussian_noise_035" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$MMSE_CKPT" \
  --results_path "$OUT_MMSE" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

2026-05-23 08:59:06.716110: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see sl

### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_NF"

!python test.py \
  --precision "32" \
  --degradation "gaussian_noise_035" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$NF_CKPT" \
  --results_path "$OUT_NF" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCY"

!python test.py \
  --precision "32" \
  --degradation "gaussian_noise_035" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCY_CKPT" \
  --results_path "$OUT_PCY" \
  --num_flow_steps 100

/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCMMSE"

!python test.py \
  --precision "32" \
  --degradation "gaussian_noise_035" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCMMSE_CKPT" \
  --results_path "$OUT_PCMMSE" \
  --num_flow_steps 100

/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

## Super-resolution

### PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PMRF"

!python test.py \
  --precision "32" \
  --degradation "sr_bicubic_x8_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PMRF_CKPT" \
  --results_path "$OUT_PMRF" \
  --num_flow_steps 100

/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

### MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_MMSE"

!python test.py \
  --precision "32" \
  --degradation "sr_bicubic_x8_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$MMSE_CKPT" \
  --results_path "$OUT_MMSE" \
  --num_flow_steps 100

/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

2026-05-23 09:57:43.385265: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see sl

### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_NF"

!python test.py \
  --precision "32" \
  --degradation "sr_bicubic_x8_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$NF_CKPT" \
  --results_path "$OUT_NF" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCY"

!python test.py \
  --precision "32" \
  --degradation "sr_bicubic_x8_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCY_CKPT" \
  --results_path "$OUT_PCY" \
  --num_flow_steps 100

/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCMMSE"

!python test.py \
  --precision "32" \
  --degradation "sr_bicubic_x8_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCMMSE_CKPT" \
  --results_path "$OUT_PCMMSE" \
  --num_flow_steps 100

/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

## Inpainting

### PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PMRF"

!python test.py \
  --precision "32" \
  --degradation "random_inpainting_gaussian_noise_01" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PMRF_CKPT" \
  --results_path "$OUT_PMRF" \
  --num_flow_steps 100

/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

### MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_MMSE"

!python test.py \
  --precision "32" \
  --degradation "random_inpainting_gaussian_noise_01" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$MMSE_CKPT" \
  --results_path "$OUT_MMSE" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

2026-05-23 10:59:52.571439: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see sl

### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_NF"

!python test.py \
  --precision "32" \
  --degradation "random_inpainting_gaussian_noise_01" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$NF_CKPT" \
  --results_path "$OUT_NF" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCY"

!python test.py \
  --precision "32" \
  --degradation "random_inpainting_gaussian_noise_01" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCY_CKPT" \
  --results_path "$OUT_PCY" \
  --num_flow_steps 100

/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCMMSE"

!python test.py \
  --precision "32" \
  --degradation "random_inpainting_gaussian_noise_01" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCMMSE_CKPT" \
  --results_path "$OUT_PCMMSE" \
  --num_flow_steps 100

/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

# Evaluation

## Colorization

### PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_zappos.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/colorization_pmrf_test_K100/colorization_gaussian_noise_025/pmrf/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [01:32<00:00,  2.32s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

### MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_zappos.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/colorization_mmse_test_K100/colorization_gaussian_noise_025/mmse/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [03:22<00:00,  5.07s/it]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_zappos.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/colorization_naive_flow_test_K100/colorization_gaussian_noise_025/naive_flow/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [03:18<00:00,  4.95s/it]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_zappos.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/colorization_post_con_on_y_test_K100/colorization_gaussian_noise_025/posterior_conditioned_on_y/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [03:23<00:00,  5.10s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_zappos.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/colorization_post_con_on_mmse_test_K100/colorization_gaussian_noise_025/posterior_conditioned_on_mmse/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [03:21<00:00,  5.03s/it]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

## Gaussian Noise

### PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_zappos.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/gaussion_pmrf_test_K100/gaussian_noise_035/pmrf/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
Downloading: "https://github.com/photosynthesis-team/photosynthesis.metrics/releases/download/v0.4.0/lpips_weights.pt" to /root/.cache/torch/hub/checkpoints/lpips_weights.pt
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100% 40/40 [05:06<00:00,  7.66s/it]
Creatin

### MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_zappos.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/gaussion_mmse_test_K100/gaussian_noise_035/mmse/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [03:27<00:00,  5.19s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

###Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_zappos.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/gaussion_naive_flow_test_K100/gaussian_noise_035/naive_flow/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [03:26<00:00,  5.17s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_zappos.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/gaussion_post_con_on_y_test_K100/gaussian_noise_035/posterior_conditioned_on_y/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [03:26<00:00,  5.17s/it]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_zappos.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/gaussion_post_con_on_mmse_test_K100/gaussian_noise_035/posterior_conditioned_on_mmse/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [03:24<00:00,  5.11s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

## Super-resolution

### PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_zappos.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/sr_pmrf_test_K100/sr_bicubic_x8_gaussian_noise_005/pmrf/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [03:36<00:00,  5.42s/it]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

### MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_zappos.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/sr_mmse_test_K100/sr_bicubic_x8_gaussian_noise_005/mmse/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [03:26<00:00,  5.15s/it]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_zappos.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/sr_naive_flow_test_K100/sr_bicubic_x8_gaussian_noise_005/naive_flow/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [03:23<00:00,  5.10s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_zappos.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/sr_post_con_on_y_test_K100/sr_bicubic_x8_gaussian_noise_005/posterior_conditioned_on_y/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
Downloading: "https://github.com/photosynthesis-team/photosynthesis.metrics/releases/download/v0.4.0/lpips_weights.pt" to /root/.cache/torch/hub/checkpoints/lpips_weights.pt
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100% 40/40 [00:26<00:00,  1.53it/s]
Creatin

###Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_zappos.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/sr_post_con_on_mmse_test_K100/sr_bicubic_x8_gaussian_noise_005/posterior_conditioned_on_mmse/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [03:22<00:00,  5.07s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

## Inpainting

### PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_zappos.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/inpainting_pmrf_test_K100/random_inpainting_gaussian_noise_01/pmrf/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [03:21<00:00,  5.03s/it]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

###MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_zappos.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/inpainting_mmse_test_K100/random_inpainting_gaussian_noise_01/mmse/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [03:22<00:00,  5.05s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

###Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_zappos.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/inpainting_naive_flow_test_K100/random_inpainting_gaussian_noise_01/naive_flow/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [03:20<00:00,  5.02s/it]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

### Poosterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_zappos.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/inpainting_post_con_on_y_test_K100/random_inpainting_gaussian_noise_01/posterior_conditioned_on_y/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [03:21<00:00,  5.03s/it]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

### Poosterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_zappos.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/inpainting_post_con_on_mmse_test_K100/random_inpainting_gaussian_noise_01/posterior_conditioned_on_mmse/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [03:20<00:00,  5.01s/it]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

# ========================================

# ========================================

# ImageNet with optimized architecture and hyperparameters

In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"   # <--- must be set before lightning imports
import matplotlib
matplotlib.use("Agg")


## Training

### Colorization

#### Stage 1

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_mmse.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run mskmguvb (0.0s)
wandb: ⣻ setting up run mskmguvb (0.0s)
wandb: ⣽ setting up run mskmguvb (0.0s)
wandb: ⣾ setting up run mskmguvb (0.0s)
wandb: ⣷ setting up run mskmguvb (0.5s)
wandb: ⣯ setting up run mskmguvb (0.5s)
wandb: ⣟ setting up run mskmguvb (0.5s)
wandb: ⡿ setting up run mskmguvb (0.5s)
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in wandb/run-20260626_165630-mskmguvb
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run desert-galaxy-111
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/

#### Stage 2

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_pmrf.sh


/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run 0wwha69m (0.0s)
wandb: ⣻ setting up run 0wwha69m (0.0s)
wandb: ⣽ setting up run 0wwha69m (0.0s)
wandb: ⣾ setting up run 0wwha69m (0.0s)
wandb: ⣷ setting up run 0wwha69m (0.5s)
wandb: ⣯ setting up run 0wwha69m (0.5s)
wandb: ⣟ setting up run 0wwha69m (0.5s)
wandb: ⡿ setting up run 0wwha69m (0.5s)
wandb: ⢿ setting up run 0wwha69m (0.5s)
wandb: ⣻ setting up run 0wwha69m (1.0s)
wandb: ⣽ setting up run 0wwha69m (1.0s)
wandb: ⣾ setting up run 0wwha69m (1.0s)
wandb: ⣷ setting up run 0wwha69m (1.0s)
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in wandb/run-202606

#### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_naive_flow.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run zws5a2nj (0.0s)
wandb: ⣻ setting up run zws5a2nj (0.0s)
wandb: ⣽ setting up run zws5a2nj (0.0s)
wandb: ⣾ setting up run zws5a2nj (0.0s)
wandb: ⣷ setting up run zws5a2nj (0.5s)
wandb: ⣯ setting up run zws5a2nj (0.5s)
wandb: ⣟ setting up run zws5a2nj (0.5s)
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in wandb/run-20260627_064303-zws5a2nj
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run glowing-butterfly-114
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wa

#### Posterior conditoined on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_y.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run 83it549s (0.0s)
wandb: ⣻ setting up run 83it549s (0.0s)
wandb: ⣽ setting up run 83it549s (0.0s)
wandb: ⣾ setting up run 83it549s (0.0s)
wandb: ⣷ setting up run 83it549s (0.5s)
wandb: ⣯ setting up run 83it549s (0.5s)
wandb: ⣟ setting up run 83it549s (0.5s)
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in wandb/run-20260627_080321-83it549s
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run cerulean-hill-116
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.

#### Posterior conditoined on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_mmse_model.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: 2
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run yk8hu833 (0.0s)
wandb: ⣻ setting up run yk8hu833 (0.0s)
wandb: ⣽ setting up run yk8hu833 (0.0s)
wan

### Gaussian Noise

#### Stage 1

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_mmse.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run pyfoyidm (0.0s)
wandb: ⣻ setting up run pyfoyidm (0.0s)
wandb: ⣽ setting up run pyfoyidm (0.0s)
wandb: ⣾ setting up run pyfoyidm (0.0s)
wandb: ⣷ setting up run pyfoyidm (0.5s)
wandb: ⣯ setting up run pyfoyidm (0.5s)
wandb: ⣟ setting up run pyfoyidm (0.5s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in wandb/run-20260620_185106-pyfoyidm
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run firm-firebrand-76
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.

#### Stage 2

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_pmrf.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run jvmq0h8n (0.0s)
wandb: ⣻ setting up run jvmq0h8n (0.0s)
wandb: ⣽ setting up run jvmq0h8n (0.0s)
wandb: ⣾ setting up run jvmq0h8n (0.0s)
wandb: ⣷ setting up run jvmq0h8n (0.5s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in wandb/run-20260620_223554-jvmq0h8n
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run bumbling-music-77
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/jvmq0h8n
wandb

#### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_naive_flow.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run o4pkzuf2 (0.0s)
wandb: ⣻ setting up run o4pkzuf2 (0.0s)
wandb: ⣽ setting up run o4pkzuf2 (0.0s)
wandb: ⣾ setting up run o4pkzuf2 (0.0s)
wandb: ⣷ setting up run o4pkzuf2 (0.5s)
wandb: ⣯ setting up run o4pkzuf2 (0.5s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in wandb/run-20260621_004028-o4pkzuf2
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run pleasant-galaxy-78
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-

#### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_y.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run n6lctnun (0.0s)
wandb: ⣻ setting up run n6lctnun (0.0s)
wandb: ⣽ setting up run n6lctnun (0.0s)
wandb: ⣾ setting up run n6lctnun (0.0s)
wandb: ⣷ setting up run n6lctnun (0.5s)
wandb: ⣯ setting up run n6lctnun (0.5s)
wandb: ⣟ setting up run n6lctnun (0.5s)
wandb: ⡿ setting up run n6lctnun (0.5s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in wandb/run-20260621_030038-n6lctnun
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run honest-planet-79
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/P

#### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_mmse_model.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run ap2klcfw (0.0s)
wandb: ⣻ setting up run ap2klcfw (0.0s)
wandb: ⣽ setting up run ap2klcfw (0.0s)
wandb: ⣾ setting up run ap2klcfw (0.0s)
wandb: ⣷ setting up run ap2klcfw (0.5s)
wandb: ⣯ setting up run ap2klcfw (0.5s)
wandb: ⣟ setting up run ap2klcfw (0.5s)
wandb: ⡿ setting up run ap2klcfw (0.5s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in wandb/run-20260621_060900-ap2klcfw
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run restful-sound-80
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/P

### Super resolution

#### Stage 1

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_mmse.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run rqjdusz7 (0.0s)
wandb: ⣻ setting up run rqjdusz7 (0.0s)
wandb: ⣽ setting up run rqjdusz7 (0.0s)
wandb: ⣾ setting up run rqjdusz7 (0.0s)
wandb: ⣷ setting up run rqjdusz7 (0.5s)
wandb: ⣯ setting up run rqjdusz7 (0.5s)
wandb: ⣟ setting up run rqjdusz7 (0.5s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in wandb/run-20260621_095903-rqjdusz7
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run fresh-sea-82
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/da

#### Stage 2

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_pmrf.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run ojbcfufh (0.0s)
wandb: ⣻ setting up run ojbcfufh (0.0s)
wandb: ⣽ setting up run ojbcfufh (0.0s)
wandb: ⣾ setting up run ojbcfufh (0.0s)
wandb: ⣷ setting up run ojbcfufh (0.5s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in wandb/run-20260621_191020-ojbcfufh
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run daily-armadillo-85
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/ojbcfufh
wand

#### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_naive_flow.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run mujrxtn0 (0.0s)
wandb: ⣻ setting up run mujrxtn0 (0.0s)
wandb: ⣽ setting up run mujrxtn0 (0.0s)
wandb: ⣾ setting up run mujrxtn0 (0.0s)
wandb: ⣷ setting up run mujrxtn0 (0.5s)
wandb: ⣯ setting up run mujrxtn0 (0.5s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in wandb/run-20260621_133920-mujrxtn0
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run eternal-voice-83
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-un

#### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_y.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run z4ilna56 (0.0s)
wandb: ⣻ setting up run z4ilna56 (0.0s)
wandb: ⣽ setting up run z4ilna56 (0.0s)
wandb: ⣾ setting up run z4ilna56 (0.0s)
wandb: ⣷ setting up run z4ilna56 (0.5s)
wandb: ⣯ setting up run z4ilna56 (0.5s)
wandb: ⣟ setting up run z4ilna56 (0.5s)
wandb: ⡿ setting up run z4ilna56 (0.5s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in wandb/run-20260621_162511-z4ilna56
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run rich-smoke-84
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF

#### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_mmse_model.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run 12t29j08 (0.0s)
wandb: ⣻ setting up run 12t29j08 (0.0s)
wandb: ⣽ setting up run 12t29j08 (0.0s)
wandb: ⣾ setting up run 12t29j08 (0.0s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in wandb/run-20260621_223604-12t29j08
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run generous-lake-86
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/12t29j08
wandb: Detected [huggingface_hub.inference] in

## Inference

In [ ]:
PMRF_CKPT = "/content/drive/MyDrive/PMRF/PMRF/new_colorization_pmrf/checkpoints/last.ckpt"   # stage-2
MMSE_CKPT = "/content/drive/MyDrive/PMRF/PMRF/new_colorization_mmse/checkpoints/last.ckpt"   # stage-1
NF_CKPT="/content/drive/MyDrive/PMRF/PMRF/new_colorization_naive_flow/checkpoints/last.ckpt"
PCY_CKPT="/content/drive/MyDrive/PMRF/PMRF/new_colorization_post_con_on_y/checkpoints/last.ckpt"
PCMMSE_CKPT="/content/drive/MyDrive/PMRF/PMRF/new_colorization_post_con_on_mmse/checkpoints/last.ckpt"

TEST_ROOT = "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test"  # <-- your flat 5k images

OUT_PMRF  = "/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_colorization_pmrf_test_K100"
OUT_MMSE  = "/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_colorization_mmse_test_K100"
OUT_NF="/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_colorization_naive_flow_test_K100"
OUT_PCY="/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_colorization_post_con_on_y_test_K100"
OUT_PCMMSE="/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_colorization_post_con_on_mmse_test_K100"

###Colorization

####MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_MMSE"

!python test.py \
  --precision "32" \
  --degradation "colorization_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$MMSE_CKPT" \
  --results_path "$OUT_MMSE" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

2026-06-28 20:11:52.471560: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see sl

####PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PMRF"

!python test.py \
  --precision "32" \
  --degradation "colorization_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PMRF_CKPT" \
  --results_path "$OUT_PMRF" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100% 91.2M/91.2M [00:01<00:00, 51.9MB/s]
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatical

####Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_NF"

!python test.py \
  --precision "32" \
  --degradation "colorization_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$NF_CKPT" \
  --results_path "$OUT_NF" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

####Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCY"

!python test.py \
  --precision "32" \
  --degradation "colorization_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCY_CKPT" \
  --results_path "$OUT_PCY" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

####Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCMMSE"

!python test.py \
  --precision "32" \
  --degradation "colorization_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCMMSE_CKPT" \
  --results_path "$OUT_PCMMSE" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

### Gaussian Noise

#### MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_MMSE"

!python test.py \
  --precision "32" \
  --degradation "gaussian_noise_02" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$MMSE_CKPT" \
  --results_path "$OUT_MMSE" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

2026-06-24 17:43:00.454598: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see sl

#### PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PMRF"

!python test.py \
  --precision "32" \
  --degradation "gaussian_noise_02" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PMRF_CKPT" \
  --results_path "$OUT_PMRF" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100% 91.2M/91.2M [00:00<00:00, 572MB/s]
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automaticall

#### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_NF"

!python test.py \
  --precision "32" \
  --degradation "gaussian_noise_02" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$NF_CKPT" \
  --results_path "$OUT_NF" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

#### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCY"

!python test.py \
  --precision "32" \
  --degradation "gaussian_noise_02" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCY_CKPT" \
  --results_path "$OUT_PCY" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

#### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCMMSE"

!python test.py \
  --precision "32" \
  --degradation "gaussian_noise_02" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCMMSE_CKPT" \
  --results_path "$OUT_PCMMSE" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

### Super Resolution

####MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_MMSE"

!python test.py \
  --precision "32" \
  --degradation "sr_bicubic_x4_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$MMSE_CKPT" \
  --results_path "$OUT_MMSE" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

2026-06-24 22:03:11.656537: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see sl

####PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PMRF"

!python test.py \
  --precision "32" \
  --degradation "sr_bicubic_x4_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PMRF_CKPT" \
  --results_path "$OUT_PMRF" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

####Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_NF"

!python test.py \
  --precision "32" \
  --degradation "sr_bicubic_x4_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$NF_CKPT" \
  --results_path "$OUT_NF" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

####Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCY"

!python test.py \
  --precision "32" \
  --degradation "sr_bicubic_x4_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCY_CKPT" \
  --results_path "$OUT_PCY" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

####Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCMMSE"

!python test.py \
  --precision "32" \
  --degradation "sr_bicubic_x4_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCMMSE_CKPT" \
  --results_path "$OUT_PCMMSE" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

## Evaluation

###Colorization

####MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_colorization_mmse_test_K100/colorization_gaussian_noise_005/mmse/xhat"

/content/drive/MyDrive/PMRF/evaluation
Downloading: "https://github.com/photosynthesis-team/photosynthesis.metrics/releases/download/v0.4.0/lpips_weights.pt" to /root/.cache/torch/hub/checkpoints/lpips_weights.pt
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100% 40/40 [01:08<00:00,  1.71s/it]
Creatin

####PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_colorization_pmrf_test_K100/colorization_gaussian_noise_005/pmrf/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:17<00:00,  2.28it/s]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

####Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_colorization_naive_flow_test_K100/colorization_gaussian_noise_005/naive_flow/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:17<00:00,  2.27it/s]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

####Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_colorization_post_con_on_y_test_K100/colorization_gaussian_noise_005/posterior_conditioned_on_y/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:17<00:00,  2.32it/s]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

####Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_colorization_post_con_on_mmse_test_K100/colorization_gaussian_noise_005/posterior_conditioned_on_mmse/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:17<00:00,  2.32it/s]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

### Gaussian Noise

#### MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_gaussian_noise_mmse_test_K100/gaussian_noise_02/mmse/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [10:16<00:00, 15.42s/it]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100% 91.2M/91.

#### PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_gaussian_noise_pmrf_test_K100/gaussian_noise_02/pmrf/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [06:14<00:00,  9.37s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

#### Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_gaussian_noise_naive_flow_test_K100/gaussian_noise_02/naive_flow/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [05:41<00:00,  8.54s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

#### Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_gaussian_noise_post_con_on_y_test_K100/gaussian_noise_02/posterior_conditioned_on_y/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [06:16<00:00,  9.41s/it]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

#### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_gaussian_noise_post_con_on_mmse_test_K100/gaussian_noise_02/posterior_conditioned_on_mmse/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [08:16<00:00, 12.41s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

### Super Resolution

####MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_super_resolution_mmse_test_K100/sr_bicubic_x4_gaussian_noise_005/mmse/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [01:09<00:00,  1.74s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

####PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_super_resolution_pmrf_test_K100/sr_bicubic_x4_gaussian_noise_005/pmrf/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [01:03<00:00,  1.59s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

####Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_super_resolution_naive_flow_test_K100/sr_bicubic_x4_gaussian_noise_005/naive_flow/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:59<00:00,  1.48s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

####Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_super_resolution_post_con_on_y_test_K100/sr_bicubic_x4_gaussian_noise_005/posterior_conditioned_on_y/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:59<00:00,  1.48s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

####Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_super_resolution_post_con_on_mmse_test_K100/sr_bicubic_x4_gaussian_noise_005/posterior_conditioned_on_mmse/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:55<00:00,  1.38s/it]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - 

# ========================================

# ========================================

# Zappos with optimized architecture and hyperparameters

In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"   # <--- must be set before lightning imports
import matplotlib
matplotlib.use("Agg")


## Training

### Colorization

####Stage 1

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_mmse.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: 2
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run gauoonzg (0.0s)
wandb: ⣻ setting up run gauoonzg (0.0s)
wandb: ⣽ setting up run gauoonzg (0.0s)
wan

####Stage 2

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_pmrf.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run r4hgbk2d (0.0s)
wandb: ⣻ setting up run r4hgbk2d (0.0s)
wandb: ⣽ setting up run r4hgbk2d (0.0s)
wandb: ⣾ setting up run r4hgbk2d (0.0s)
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in wandb/run-20260628_090207-r4hgbk2d
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run ancient-yogurt-126
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/r4hgbk2d
wandb: Detected [huggingface_hub.inference] 

####Naive FLow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_naive_flow.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run y6iwdvct (0.0s)
wandb: ⣻ setting up run y6iwdvct (0.0s)
wandb: ⣽ setting up run y6iwdvct (0.0s)
wandb: ⣾ setting up run y6iwdvct (0.0s)
wandb: ⣷ setting up run y6iwdvct (0.5s)
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in wandb/run-20260628_022802-y6iwdvct
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run fragrant-field-122
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/y6iwdvct
wand

####Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_y.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run s9kwcs31 (0.0s)
wandb: ⣻ setting up run s9kwcs31 (0.0s)
wandb: ⣽ setting up run s9kwcs31 (0.0s)
wandb: ⣾ setting up run s9kwcs31 (0.0s)
wandb: ⣷ setting up run s9kwcs31 (0.5s)
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in wandb/run-20260628_073345-s9kwcs31
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run sweet-night-125
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/s9kwcs31
wandb: 

#### Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_mmse_model.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run wyqseh2f (0.0s)
wandb: ⣻ setting up run wyqseh2f (0.0s)
wandb: ⣽ setting up run wyqseh2f (0.0s)
wandb: ⣾ setting up run wyqseh2f (0.0s)
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in wandb/run-20260628_112939-wyqseh2f
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run royal-fog-127
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/wyqseh2f
wandb: Detected [huggingface_hub.inference] in us

### Gaussian Noise

####Stage 1

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_mmse.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run zxj3nb4m (0.0s)
wandb: ⣻ setting up run zxj3nb4m (0.0s)
wandb: ⣽ setting up run zxj3nb4m (0.0s)
wandb: ⣾ setting up run zxj3nb4m (0.0s)
wandb: ⣷ setting up run zxj3nb4m (0.5s)
wandb: ⣯ setting up run zxj3nb4m (0.5s)
wandb: ⣟ setting up run zxj3nb4m (0.5s)
wandb: ⡿ setting up run zxj3nb4m (0.5s)
wandb: ⢿ setting up run zxj3nb4m (0.5s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in wandb/run-20260622_070527-zxj3nb4m
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run spring-snow-91
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2

####Stage 2

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_pmrf.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run uh0pagx4 (0.0s)
wandb: ⣻ setting up run uh0pagx4 (0.0s)
wandb: ⣽ setting up run uh0pagx4 (0.0s)
wandb: ⣾ setting up run uh0pagx4 (0.0s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in wandb/run-20260622_101636-uh0pagx4
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run zany-morning-93
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/uh0pagx4
wandb: Detected [huggingface_hub.inference] in 

####Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_naive_flow.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run mzmoyipl (0.0s)
wandb: ⣻ setting up run mzmoyipl (0.0s)
wandb: ⣽ setting up run mzmoyipl (0.0s)
wandb: ⣾ setting up run mzmoyipl (0.0s)
wandb: ⣷ setting up run mzmoyipl (0.5s)
wandb: ⣯ setting up run mzmoyipl (0.5s)
wandb: ⣟ setting up run mzmoyipl (0.5s)
wandb: ⡿ setting up run mzmoyipl (0.5s)
wandb: ⢿ setting up run mzmoyipl (0.5s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in wandb/run-20260622_155010-mzmoyipl
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run copper-grass-94
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov

####Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_y.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run 3lwv1ln1 (0.0s)
wandb: ⣻ setting up run 3lwv1ln1 (0.0s)
wandb: ⣽ setting up run 3lwv1ln1 (0.0s)
wandb: ⣾ setting up run 3lwv1ln1 (0.0s)
wandb: ⣷ setting up run 3lwv1ln1 (0.5s)
wandb: ⣯ setting up run 3lwv1ln1 (0.5s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in wandb/run-20260622_190501-3lwv1ln1
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run polished-hill-95
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-un

####Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_mmse_model.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run ga1ssrhz (0.0s)
wandb: ⣻ setting up run ga1ssrhz (0.0s)
wandb: ⣽ setting up run ga1ssrhz (0.0s)
wandb: ⣾ setting up run ga1ssrhz (0.0s)
wandb: ⣷ setting up run ga1ssrhz (0.5s)
wandb: ⣯ setting up run ga1ssrhz (0.5s)
wandb: ⣟ setting up run ga1ssrhz (0.5s)
wandb: ⡿ setting up run ga1ssrhz (0.5s)
wandb: ⢿ setting up run ga1ssrhz (0.5s)
wandb: ⣻ setting up run ga1ssrhz (1.0s)
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in wandb/run-20260623_071732-ga1ssrhz
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run crimson-wildflower-100
wandb: ⭐️ V

### Super Resolution

####Stage 1

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_mmse.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run l1lrityq (0.0s)
wandb: ⣻ setting up run l1lrityq (0.0s)
wandb: ⣽ setting up run l1lrityq (0.0s)
wandb: ⣾ setting up run l1lrityq (0.0s)
wandb: ⣷ setting up run l1lrityq (0.5s)
wandb: ⣯ setting up run l1lrityq (0.5s)
wandb: ⣟ setting up run l1lrityq (0.5s)
wandb: ⡿ setting up run l1lrityq (0.5s)
wandb: ⢿ setting up run l1lrityq (0.5s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in wandb/run-20260623_044410-l1lrityq
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run spring-salad-98
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov

####Stage 2

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_pmrf.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run 11i1nvxm (0.0s)
wandb: ⣻ setting up run 11i1nvxm (0.0s)
wandb: ⣽ setting up run 11i1nvxm (0.0s)
wandb: ⣾ setting up run 11i1nvxm (0.0s)
wandb: ⣷ setting up run 11i1nvxm (0.5s)
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in wandb/run-20260623_125235-11i1nvxm
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run generous-dust-102
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/11i1nvxm
wandb

####Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_naive_flow.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: 2
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run zzcpfyeh (0.0s)
wandb: ⣻ setting up run zzcpfyeh (0.0s)
wandb: ⣽ setting up run zzcpfyeh (0.0s)
wan

####Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_y.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run d3p523su (0.0s)
wandb: ⣻ setting up run d3p523su (0.0s)
wandb: ⣽ setting up run d3p523su (0.0s)
wandb: ⣾ setting up run d3p523su (0.0s)
wandb: ⣷ setting up run d3p523su (0.5s)
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in wandb/run-20260623_214628-d3p523su
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run daily-voice-104
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF
wandb: 🚀 View run at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz/PMRF/runs/d3p523su
wandb: 

####Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!bash train_posterior_conditioned_on_mmse_model.sh

/content/drive/MyDrive/PMRF
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: daniel-morgunov2002 (daniel-morgunov2002-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run ih6xfk3o (0.0s)
wandb: ⣻ setting up run ih6xfk3o (0.0s)
wandb: ⣽ setting up run ih6xfk3o (0.0s)
wandb: ⣾ setting up run ih6xfk3o (0.0s)
wandb: ⣷ setting up run ih6xfk3o (0.5s)
wandb: ⣯ setting up run ih6xfk3o (0.5s)
wandb: ⣟ setting up run ih6xfk3o (0.5s)
wandb: ⡿ setting up run ih6xfk3o (0.5s)
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in wandb/run-20260624_091902-ih6xfk3o
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run ethereal-dream-107
wandb: ⭐️ View project at https://wandb.ai/daniel-morgunov2002-johannes-kepler-universit-t-linz

## Inference

In [ ]:
PMRF_CKPT = "/content/drive/MyDrive/PMRF/PMRF/new_zap_colorization_pmrf/checkpoints/last.ckpt"   # stage-2
MMSE_CKPT = "/content/drive/MyDrive/PMRF/PMRF/new_zap_colorization_mmse/checkpoints/last.ckpt"   # stage-1
NF_CKPT="/content/drive/MyDrive/PMRF/PMRF/new_zap_colorization_naive_flow/checkpoints/last.ckpt"
PCY_CKPT="/content/drive/MyDrive/PMRF/PMRF/new_zap_colorization_post_con_on_y/checkpoints/last.ckpt"
PCMMSE_CKPT="/content/drive/MyDrive/PMRF/PMRF/new_zap_colorization_post_con_on_mmse/checkpoints/last.ckpt"

TEST_ROOT = "/content/drive/MyDrive/PMRF/data/zap50k_128/test"  # <-- your flat 5k images

OUT_PMRF  = "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_colorization_pmrf_test_K100"
OUT_MMSE  = "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_colorization_mmse_test_K100"
OUT_NF="/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_colorization_naive_flow_test_K100"
OUT_PCY="/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_colorization_post_con_on_y_test_K100"
OUT_PCMMSE="/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_colorization_post_con_on_mmse_test_K100"

###Colorization

####MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_MMSE"

!python test.py \
  --precision "32" \
  --degradation "colorization_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$MMSE_CKPT" \
  --results_path "$OUT_MMSE" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

2026-06-28 21:59:35.821074: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see sl

####PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PMRF"

!python test.py \
  --precision "32" \
  --degradation "colorization_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PMRF_CKPT" \
  --results_path "$OUT_PMRF" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

####Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_NF"

!python test.py \
  --precision "32" \
  --degradation "colorization_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$NF_CKPT" \
  --results_path "$OUT_NF" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

####Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCY"

!python test.py \
  --precision "32" \
  --degradation "colorization_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCY_CKPT" \
  --results_path "$OUT_PCY" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100% 91.2M/91.2M [00:00<00:00, 375MB/s]
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automaticall

####Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCMMSE"

!python test.py \
  --precision "32" \
  --degradation "colorization_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCMMSE_CKPT" \
  --results_path "$OUT_PCMMSE" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

### Gaussian Noise

####MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_MMSE"

!python test.py \
  --precision "32" \
  --degradation "gaussian_noise_02" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$MMSE_CKPT" \
  --results_path "$OUT_MMSE" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

2026-06-25 08:47:47.851790: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see sl

####PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PMRF"

!python test.py \
  --precision "32" \
  --degradation "gaussian_noise_02" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PMRF_CKPT" \
  --results_path "$OUT_PMRF" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100% 91.2M/91.2M [00:00<00:00, 323MB/s]
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automaticall

####Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_NF"

!python test.py \
  --precision "32" \
  --degradation "gaussian_noise_02" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$NF_CKPT" \
  --results_path "$OUT_NF" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

####Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCY"

!python test.py \
  --precision "32" \
  --degradation "gaussian_noise_02" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCY_CKPT" \
  --results_path "$OUT_PCY" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

####Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCMMSE"

!python test.py \
  --precision "32" \
  --degradation "gaussian_noise_02" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCMMSE_CKPT" \
  --results_path "$OUT_PCMMSE" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

### Super Resolution

####MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_MMSE"

!python test.py \
  --precision "32" \
  --degradation "sr_bicubic_x4_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$MMSE_CKPT" \
  --results_path "$OUT_MMSE" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

2026-06-25 10:45:42.379345: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see sl

####PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PMRF"

!python test.py \
  --precision "32" \
  --degradation "sr_bicubic_x4_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PMRF_CKPT" \
  --results_path "$OUT_PMRF" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

####Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_NF"

!python test.py \
  --precision "32" \
  --degradation "sr_bicubic_x4_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$NF_CKPT" \
  --results_path "$OUT_NF" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

####Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCY"

!python test.py \
  --precision "32" \
  --degradation "sr_bicubic_x4_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCY_CKPT" \
  --results_path "$OUT_PCY" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

####Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF
!mkdir -p "$OUT_PCMMSE"

!python test.py \
  --precision "32" \
  --degradation "sr_bicubic_x4_gaussian_noise_005" \
  --test_data_root "$TEST_ROOT" \
  --num_gpus 1 \
  --batch_size 64 \
  --num_workers 5 \
  --img_size 128 \
  --ckpt_path "$PCMMSE_CKPT" \
  --results_path "$OUT_PCMMSE" \
  --num_flow_steps 100


/content/drive/MyDrive/PMRF
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes 

##Evaluation

###Colorization

####MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_colorization_mmse_test_K100/colorization_gaussian_noise_005/mmse/xhat"

/content/drive/MyDrive/PMRF/evaluation
Downloading: "https://github.com/photosynthesis-team/photosynthesis.metrics/releases/download/v0.4.0/lpips_weights.pt" to /root/.cache/torch/hub/checkpoints/lpips_weights.pt
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100% 40/40 [06:06<00:00,  9.15s/it]
Creatin

####PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_colorization_pmrf_test_K100/colorization_gaussian_noise_005/pmrf/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [05:52<00:00,  8.81s/it]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

####Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_colorization_naive_flow_test_K100/colorization_gaussian_noise_005/naive_flow/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [05:50<00:00,  8.77s/it]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

####Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_colorization_post_con_on_y_test_K100/colorization_gaussian_noise_005/posterior_conditioned_on_y/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:43<00:00,  1.09s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

####Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_colorization_post_con_on_mmse_test_K100/colorization_gaussian_noise_005/posterior_conditioned_on_mmse/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:13<00:00,  3.07it/s]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

### Gaussian Noise

####MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_gaussian_noise_mmse_test_K100/gaussian_noise_02/mmse/xhat"

/content/drive/MyDrive/PMRF/evaluation
Downloading: "https://github.com/photosynthesis-team/photosynthesis.metrics/releases/download/v0.4.0/lpips_weights.pt" to /root/.cache/torch/hub/checkpoints/lpips_weights.pt
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100% 40/40 [01:02<00:00,  1.56s/it]
Creatin

####PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_gaussian_noise_pmrf_test_K100/gaussian_noise_02/pmrf/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:17<00:00,  2.24it/s]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

####Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_gaussian_noise_naive_flow_test_K100/gaussian_noise_02/naive_flow/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:17<00:00,  2.32it/s]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

####Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_gaussian_noise_post_con_on_y_test_K100/gaussian_noise_02/posterior_conditioned_on_y/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:18<00:00,  2.21it/s]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

####Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_gaussian_noise_post_con_on_mmse_test_K100/gaussian_noise_02/posterior_conditioned_on_mmse/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:18<00:00,  2.21it/s]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

### Super Resolution

####MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_super_resolution_mmse_test_K100/sr_bicubic_x4_gaussian_noise_005/mmse/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [01:09<00:00,  1.74s/it]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

####PMRF

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_super_resolution_pmrf_test_K100/sr_bicubic_x4_gaussian_noise_005/pmrf/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [01:03<00:00,  1.58s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

####Naive Flow

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_super_resolution_naive_flow_test_K100/sr_bicubic_x4_gaussian_noise_005/naive_flow/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [01:01<00:00,  1.55s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

####Posterior conditioned on Y

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_super_resolution_post_con_on_y_test_K100/sr_bicubic_x4_gaussian_noise_005/posterior_conditioned_on_y/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [01:01<00:00,  1.54s/it]
Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

####Posterior conditioned on MMSE

In [ ]:
%cd /content/drive/MyDrive/PMRF/evaluation

!python compute_metrics_imagenet.py \
  --gt_path  "$TEST_ROOT" \
  --rec_path "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_super_resolution_post_con_on_mmse_test_K100/sr_bicubic_x4_gaussian_noise_005/posterior_conditioned_on_mmse/num_flow_steps=100/xhat"

/content/drive/MyDrive/PMRF/evaluation
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100% 40/40 [00:59<00:00,  1.50s/it]
Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "/content/drive/MyDrive/PMRF/data/zap50k_128/test" with extensions png,jpg,jpeg
Found 5000 samples, some are lossy-compressed - this may affect m

# Checks

In [ ]:
import os, glob, pathlib

# folders you want to inspect  (add/remove paths as needed)
folders = [
    #"/content/drive/MyDrive/PMRF/data/cropped_faces/ffhq256",
    #"/content/drive/MyDrive/PMRF/data/cropped_faces/ffhq512",
    #"/content/drive/MyDrive/PMRF/data/celeba_512_validation",
    #"/content/drive/MyDrive/PMRF/data/celeba_512_validation_lq",
    #"/content/drive/MyDrive/PMRF/data/celeba_256_test",
    #"/content/drive/MyDrive/PMRF/data/lfw-Test",
    #"/content/drive/MyDrive/PMRF/outputs/celeba512_pmrf/restored_images",
    #"/content/drive/MyDrive/PMRF/outputs/celeba512_pmrf/restored_images_posterior_mean",
    #"/content/drive/MyDrive/PMRF/outputs/lfw_pmrf/restored_images",
    #"/content/drive/MyDrive/PMRF/outputs/lfw_pmrf/restored_images_posterior_mean",
    #"/content/drive/MyDrive/PMRF/data/Wider-Test",
    #"/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/train",
    #"/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/val",
    #"/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test",
    #"/content/drive/MyDrive/PMRF/data/zap50k_128/train",
    #"/content/drive/MyDrive/PMRF/data/zap50k_128/val",
    #"/content/drive/MyDrive/PMRF/data/zap50k_128/test"
    #"/content/drive/MyDrive/PMRF/outputs/colorization_pmrf_test_K100/colorization_gaussian_noise_025/pmrf/num_flow_steps=100/xhat",
    #"/content/drive/MyDrive/PMRF/outputs/colorization_mmse_test_K100/colorization_gaussian_noise_025/mmse/xhat",
    #"/content/drive/MyDrive/PMRF/outputs/zappos_data/colorization_mmse_test_K100/colorization_gaussian_noise_025/mmse/xhat",
    #"/content/drive/MyDrive/PMRF/outputs/zappos_data/colorization_naive_flow_test_K100/colorization_gaussian_noise_025/naive_flow/num_flow_steps=100/xhat",
    #"/content/drive/MyDrive/PMRF/outputs/zappos_data/colorization_pmrf_test_K100/colorization_gaussian_noise_025/pmrf/num_flow_steps=100/xhat",
    #"/content/drive/MyDrive/PMRF/outputs/zappos_data/colorization_post_con_on_mmse_test_K100/colorization_gaussian_noise_025/posterior_conditioned_on_mmse/num_flow_steps=100/xhat",
    #"/content/drive/MyDrive/PMRF/outputs/zappos_data/colorization_post_con_on_y_test_K100/colorization_gaussian_noise_025/posterior_conditioned_on_y/num_flow_steps=100/xhat",
    #"/content/drive/MyDrive/PMRF/outputs/zappos_data/gaussion_mmse_test_K100/gaussian_noise_035/mmse/xhat",
    #"/content/drive/MyDrive/PMRF/outputs/zappos_data/gaussion_naive_flow_test_K100/gaussian_noise_035/naive_flow/num_flow_steps=100/xhat",
    #"/content/drive/MyDrive/PMRF/outputs/zappos_data/gaussion_pmrf_test_K100/gaussian_noise_035/pmrf/num_flow_steps=100/xhat",
    #"/content/drive/MyDrive/PMRF/outputs/zappos_data/gaussion_post_con_on_mmse_test_K100/gaussian_noise_035/posterior_conditioned_on_mmse/num_flow_steps=100/xhat",
    #"/content/drive/MyDrive/PMRF/outputs/zappos_data/gaussion_post_con_on_y_test_K100/gaussian_noise_035/posterior_conditioned_on_y/num_flow_steps=100/xhat",
    #"/content/drive/MyDrive/PMRF/outputs/zappos_data/sr_mmse_test_K100/sr_bicubic_x8_gaussian_noise_005/mmse/xhat",
    #"/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_super_resolution_mmse_test_K100",
    #"/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_super_resolution_pmrf_test_K100",
    #"/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_super_resolution_naive_flow_test_K100",
    #"/content/drive/MyDrive/PMRF/outputs/imagenet_data/new_super_resolution_post_con_on_y_test_K100",
    "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_gaussian_noise_naive_flow_test_K100",
    "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_gaussian_noise_mmse_test_K100",
    "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_gaussian_noise_pmrf_test_K100"

    ]

img_ext = ("*.png", "*.jpg", "*.jpeg", "*.webp", "*.JPEG")  # extend if you have other formats

for folder in folders:
    folder = pathlib.Path(folder)
    if not folder.exists():
        print(f"{folder}  —  (not found)")
        continue

    # count all files matching the extensions, in this folder *and* sub-folders
    n_imgs = sum(len(glob.glob(str(folder / "**" / ext), recursive=True))
                 for ext in img_ext)
    print(f"{folder.relative_to('/content/drive/MyDrive/PMRF')}  →  {n_imgs:,} images")


outputs/zappos_data/new_zap_gaussian_noise_naive_flow_test_K100  →  15,000 images
outputs/zappos_data/new_zap_gaussian_noise_mmse_test_K100  →  10,000 images
outputs/zappos_data/new_zap_gaussian_noise_pmrf_test_K100  →  20,000 images


In [ ]:
import os, glob, pathlib

ROOT = pathlib.Path("/content/drive/MyDrive/PMRF/outputs/zappos_data")
img_ext = ("*.png", "*.jpg", "*.jpeg", "*.webp", "*.JPEG")

if not ROOT.exists():
    print(f"{ROOT} — (not found)")
else:
    # iterate over subdirectories inside imagenet_data
    for subfolder in sorted(ROOT.iterdir()):
        if subfolder.is_dir():
            # count images recursively inside this subfolder
            n_imgs = sum(
                len(glob.glob(str(subfolder / "**" / ext), recursive=True))
                for ext in img_ext
            )
            print(f"{subfolder.relative_to(ROOT)}  →  {n_imgs:,} images")

            # optional: list how many in each sub-subfolder (e.g. y, xhat, mmse_samples)
            for subsub in sorted(subfolder.iterdir()):
                if subsub.is_dir():
                    n_subsub = sum(
                        len(glob.glob(str(subsub / "**" / ext), recursive=True))
                        for ext in img_ext
                    )
                    print(f"   └── {subsub.name:<30} {n_subsub:,} images")


colorization_mmse_test_K100  →  10,000 images
   └── colorization_gaussian_noise_025 10,000 images
colorization_naive_flow_test_K100  →  15,000 images
   └── colorization_gaussian_noise_025 15,000 images
colorization_pmrf_test_K100  →  20,000 images
   └── colorization_gaussian_noise_025 20,000 images
colorization_post_con_on_mmse_test_K100  →  20,000 images
   └── colorization_gaussian_noise_025 20,000 images
colorization_post_con_on_y_test_K100  →  15,000 images
   └── colorization_gaussian_noise_025 15,000 images
gaussion_mmse_test_K100  →  10,000 images
   └── gaussian_noise_035             10,000 images
gaussion_naive_flow_test_K100  →  15,000 images
   └── gaussian_noise_035             15,000 images
gaussion_pmrf_test_K100  →  20,000 images
   └── gaussian_noise_035             20,000 images
gaussion_post_con_on_mmse_test_K100  →  20,000 images
   └── gaussian_noise_035             20,000 images
gaussion_post_con_on_y_test_K100  →  15,000 images
   └── gaussian_noise_035        

# Visualizations

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings


# --- image rows (left column = labels) ---
rows = [
    ("Ground Truth",         Path("/content/drive/MyDrive/PMRF/visualizations/gt.png")),
    ("Degraded",             Path("/content/drive/MyDrive/PMRF/visualizations/col_deg.png")),
    ("MMSE",                 Path("/content/drive/MyDrive/PMRF/visualizations/col_mmse.png")),
    ("Naive Flow",           Path("/content/drive/MyDrive/PMRF/visualizations/col_nf.png")),
    ("PMRF",                 Path("/content/drive/MyDrive/PMRF/visualizations/col_pmrf.png")),
    ("Post. cond.\non MMSE", Path("/content/drive/MyDrive/PMRF/visualizations/col_post_mmse.png")),
    ("Post. cond.\non Y",    Path("/content/drive/MyDrive/PMRF/visualizations/col_post_y.png")),
]

def open_img(p: Path):
    if not p.exists():
        warnings.warn(f"Missing file: {p}")
        return Image.new("RGB", (256,256), "white")
    return Image.open(p).convert("RGB")

# --- figure with 2 columns: [labels | big image] ---
n = len(rows)
fig = plt.figure(figsize=(11, 12))
gs  = gridspec.GridSpec(nrows=n, ncols=2, width_ratios=[1, 11], hspace=0.18, wspace=0.05)

for i, (label, img_path) in enumerate(rows):
    # label cell (left)
    ax_lab = fig.add_subplot(gs[i, 0])
    ax_lab.axis("off")
    ax_lab.text(
        0.98, 0.5, label,
        va="center", ha="right",
        fontsize=12, fontweight="semibold", color="#222",
        linespacing=1.15,
    )

    # image cell (right)
    ax_img = fig.add_subplot(gs[i, 1])
    ax_img.imshow(open_img(img_path))
    ax_img.axis("off")

# title and margins
fig.suptitle("Colorization – ImageNet", fontsize=17, fontweight="bold", y=0.98)
fig.subplots_adjust(left=0.08, right=0.99, top=0.94, bottom=0.03)

plt.show()

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings


# --- image rows (left column = labels) ---
rows = [
    ("Ground Truth",         Path("/content/drive/MyDrive/PMRF/visualizations/gt.png")),
    ("Degraded",             Path("/content/drive/MyDrive/PMRF/visualizations/gauss_deg.png")),
    ("MMSE",                 Path("/content/drive/MyDrive/PMRF/visualizations/gauss_mmse.png")),
    ("Naive Flow",           Path("/content/drive/MyDrive/PMRF/visualizations/gauss_nf.png")),
    ("PMRF",                 Path("/content/drive/MyDrive/PMRF/visualizations/gauss_pmrf.png")),
    ("Post. cond.\non MMSE", Path("/content/drive/MyDrive/PMRF/visualizations/gauss_post_mmse.png")),
    ("Post. cond.\non Y",    Path("/content/drive/MyDrive/PMRF/visualizations/gauss_post_y.png")),
]

def open_img(p: Path):
    if not p.exists():
        warnings.warn(f"Missing file: {p}")
        return Image.new("RGB", (256,256), "white")
    return Image.open(p).convert("RGB")

# --- figure with 2 columns: [labels | big image] ---
n = len(rows)
fig = plt.figure(figsize=(11, 12))
gs  = gridspec.GridSpec(nrows=n, ncols=2, width_ratios=[1, 11], hspace=0.18, wspace=0.05)

for i, (label, img_path) in enumerate(rows):
    # label cell (left)
    ax_lab = fig.add_subplot(gs[i, 0])
    ax_lab.axis("off")
    ax_lab.text(
        0.98, 0.5, label,
        va="center", ha="right",
        fontsize=12, fontweight="semibold", color="#222",
        linespacing=1.15,
    )

    # image cell (right)
    ax_img = fig.add_subplot(gs[i, 1])
    ax_img.imshow(open_img(img_path))
    ax_img.axis("off")

# title and margins
fig.suptitle("Gaussian Noise – ImageNet", fontsize=17, fontweight="bold", y=0.98)
fig.subplots_adjust(left=0.08, right=0.99, top=0.94, bottom=0.03)

plt.show()

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings


# --- image rows (left column = labels) ---
rows = [
    ("Ground Truth",         Path("/content/drive/MyDrive/PMRF/visualizations/gt.png")),
    ("Degraded",             Path("/content/drive/MyDrive/PMRF/visualizations/sr_deg.png")),
    ("MMSE",                 Path("/content/drive/MyDrive/PMRF/visualizations/sr_mmse.png")),
    ("Naive Flow",           Path("/content/drive/MyDrive/PMRF/visualizations/sr_nf.png")),
    ("PMRF",                 Path("/content/drive/MyDrive/PMRF/visualizations/sr_pmrf.png")),
    ("Post. cond.\non MMSE", Path("/content/drive/MyDrive/PMRF/visualizations/sr_post_mmse.png")),
    ("Post. cond.\non Y",    Path("/content/drive/MyDrive/PMRF/visualizations/sr_post_y.png")),
]

def open_img(p: Path):
    if not p.exists():
        warnings.warn(f"Missing file: {p}")
        return Image.new("RGB", (256,256), "white")
    return Image.open(p).convert("RGB")

# --- figure with 2 columns: [labels | big image] ---
n = len(rows)
fig = plt.figure(figsize=(11, 12))
gs  = gridspec.GridSpec(nrows=n, ncols=2, width_ratios=[1, 11], hspace=0.18, wspace=0.05)

for i, (label, img_path) in enumerate(rows):
    # label cell (left)
    ax_lab = fig.add_subplot(gs[i, 0])
    ax_lab.axis("off")
    ax_lab.text(
        0.98, 0.5, label,
        va="center", ha="right",
        fontsize=12, fontweight="semibold", color="#222",
        linespacing=1.15,
    )

    # image cell (right)
    ax_img = fig.add_subplot(gs[i, 1])
    ax_img.imshow(open_img(img_path))
    ax_img.axis("off")

# title and margins
fig.suptitle("Super Resolution – ImageNet", fontsize=17, fontweight="bold", y=0.98)
fig.subplots_adjust(left=0.08, right=0.99, top=0.94, bottom=0.03)

plt.show()

In [ ]:
"""
Controlled face restoration grid (appendix figures).

Same layout as the practical-work figures: three single rows (ground truth,
degraded, posterior mean), then one K=5 / K=100 pair per flow method, with a
dashed separator between groups.

Differences from the practical-work version:
  * method order now matches the thesis  -> PMRF, Naive Flow, Flow cond. on Y,
    Flow cond. on X-hat*   (previously Flow-from-Y came last)
  * "Flow from Y" renamed to "Naive Flow", matching Chapter 3 onwards
  * labels use the thesis notation:  Y,  X-hat*
  * saves a PDF instead of calling plt.show()
  * images are embedded at their native 256x256 resolution
    (imshow(..., interpolation="none")). Previously matplotlib upscaled every
    image to 600x600 px with a smoothing filter (CELL_IN=2.0 at dpi=300),
    which made each grid ~30 MB and showed interpolated rather than exact
    output pixels when zoomed in.

Edit TASK, ROOT and INDICES, then run.
"""

from pathlib import Path
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.font_manager as fm
from PIL import Image

# =============================== CONFIG ===============================

TASK = "sr_bicubic_x8_gaussian_noise_005"
ROOT = Path("/content/drive/MyDrive/PMRF/controlled_experiments_results")
GT   = Path("/content/drive/MyDrive/PMRF/data/celeba_256_test")

INDICES = [1, 500, 1000, 1500, 2000, 2952, 2989, 2999]

OUTFILE  = "grid_face_superresolution.pdf"
FONTSIZE = 12
CELL_IN  = 2.0      # width of one image column, in inches (layout only;
                    # no longer affects the embedded image resolution)

# ======================================================================


def _k(k, method, sub):
    """Path to one method's outputs at a given number of flow steps."""
    return ROOT / f"num_flow_steps_{k}" / TASK / method / f"num_flow_steps={k}" / sub


# --- single rows, shown once ------------------------------------------------
SINGLE = [
    ("Ground truth",                GT),
    ("Degraded $Y$",                ROOT / f"num_flow_steps_5/{TASK}/pmrf/y"),
    ("Posterior mean $\\hat{X}^*$", ROOT / f"num_flow_steps_5/{TASK}/mmse/xhat"),
]

# --- paired rows: (label, K=5 folder, K=100 folder) --------------------------
# Order follows the thesis: PMRF, Naive Flow, then the two conditioned flows.
PAIRED = [
    ("PMRF",                        _k(5, "pmrf", "xhat"),
                                    _k(100, "pmrf", "xhat")),
    ("Naive Flow",                  _k(5, "naive_flow", "xhat"),
                                    _k(100, "naive_flow", "xhat")),
    ("Flow cond. on $Y$",           _k(5, "posterior_conditioned_on_y", "xhat"),
                                    _k(100, "posterior_conditioned_on_y", "xhat")),
    ("Flow cond. on $\\hat{X}^*$",  _k(5, "posterior_conditioned_on_mmse", "xhat"),
                                    _k(100, "posterior_conditioned_on_mmse", "xhat")),
]


def _use_latex_like_font():
    for f in ["/usr/share/texmf/fonts/opentype/public/lm/lmroman10-regular.otf",
              "/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf"]:
        if Path(f).exists():
            try:
                fm.fontManager.addfont(f)
            except Exception:
                pass
    plt.rcParams.update({"font.family": "serif",
                         "font.serif": ["Latin Modern Roman", "DejaVu Serif"],
                         "mathtext.fontset": "cm"})


def open_img(folder: Path, idx: int):
    matches = list(Path(folder).glob(f"{idx:08d}.*"))
    if not matches:
        raise FileNotFoundError(f"no file for index {idx} in {folder}")
    return Image.open(matches[0]).convert("RGB")


def build(outfile=OUTFILE, indices=INDICES, fontsize=FONTSIZE, cell_in=CELL_IN):
    _use_latex_like_font()

    n_col = len(indices)
    # 3 single rows, then for each pair: separator + 2 rows
    img_h, sep_h = 0.9, 0.30
    height_ratios = [img_h] * 3
    for _ in PAIRED:
        height_ratios += [sep_h, img_h, img_h]
    n_row = len(height_ratios)

    # margins used by subplots_adjust below; the figure size is derived from
    # them so that every image cell comes out exactly square.
    L, R, B, T = 0.085, 0.995, 0.005, 0.995
    sep_ratio = sep_h / img_h                      # separator height, in cell units
    grid_h_cells = 3 + len(PAIRED) * (sep_ratio + 2)
    fig_w = cell_in * n_col / (R - L)
    fig_h = cell_in * grid_h_cells / (T - B)
    fig = plt.figure(figsize=(fig_w, fig_h))
    gs = gridspec.GridSpec(n_row, n_col, height_ratios=height_ratios,
                           wspace=0, hspace=0.0)

    # row index in the gridspec for every image row, in drawing order
    single_rows = [0, 1, 2]
    pair_rows, sep_rows, r = [], [], 3
    for _ in PAIRED:
        sep_rows.append(r)
        pair_rows.append((r + 1, r + 2))
        r += 3

    def draw(row, folder):
        for c, idx in enumerate(indices):
            ax = fig.add_subplot(gs[row, c])
            # interpolation="none": embed the original pixels in the PDF
            # instead of a resampled (smoothed, upscaled) copy
            try:
                ax.imshow(open_img(folder, idx), interpolation="none")
            except FileNotFoundError:
                warnings.warn(f"missing index {idx} in {folder}")
                ax.imshow(Image.new("RGB", (256, 256), "white"),
                          interpolation="none")
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_visible(False)

    for row, (_, folder) in zip(single_rows, SINGLE):
        draw(row, folder)
    for (r5, r100), (_, f5, f100) in zip(pair_rows, PAIRED):
        draw(r5, f5)
        draw(r100, f100)

    for sr in sep_rows:
        ax = fig.add_subplot(gs[sr, :])
        ax.plot([0, 1], [0.5, 0.5], linestyle="--", color="black", lw=0.9)
        ax.set_xlim(0, 1); ax.axis("off")

    fig.subplots_adjust(left=L, right=R, top=T, bottom=B,
                        wspace=0.0, hspace=0.0)

    def label_at(rows_, text, x):
        b0 = gs[rows_[0], 0].get_position(fig)
        b1 = gs[rows_[-1], 0].get_position(fig)
        fig.text(x, (b0.y1 + b1.y0) / 2, text, rotation=90,
                 va="center", ha="center", fontsize=fontsize)

    for row, (label, _) in zip(single_rows, SINGLE):
        label_at([row], label, 0.042)
    for (r5, r100), (label, _, _) in zip(pair_rows, PAIRED):
        label_at([r5, r100], label, 0.030)      # group label, outer column
        label_at([r5],  "$K=5$",   0.068)       # per-row K label, inner column
        label_at([r100], "$K=100$", 0.068)

    # text, labels and separators stay vector graphics; with
    # interpolation="none" the dpi no longer changes the embedded images
    fig.savefig(outfile, bbox_inches="tight", pad_inches=0.01, dpi=300)
    plt.close(fig)
    print(f"wrote {outfile}  ({n_row} gridspec rows, {n_col} columns)")
    return outfile


if __name__ == "__main__":
    build()

wrote grid_face_superresolution.pdf  (15 gridspec rows, 8 columns)


In [ ]:
"""
Build a labelled comparison grid (rows = methods, columns = samples) as a PDF
for inclusion in the thesis.

Usage in Colab: edit the CONFIG block, then run the cell.
Requires only matplotlib + pillow (both preinstalled in Colab).
"""

import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from PIL import Image

# =============================== CONFIG ===============================

# Each entry: ("row label shown on the left", "/path/to/folder")
# Order top-to-bottom. Delete or reorder rows freely.
ROWS = [
    ("Ground truth",                "/content/drive/MyDrive/PMRF/data/zap50k_128/test"),
    ("Degraded $Y$",                "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_super_resolution_pmrf_test_K100/sr_bicubic_x4_gaussian_noise_005/pmrf/y"),
    ("Posterior mean $\\hat{X}^*$", "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_super_resolution_mmse_test_K100/sr_bicubic_x4_gaussian_noise_005/mmse/xhat"),
    ("PMRF",                        "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_super_resolution_pmrf_test_K100/sr_bicubic_x4_gaussian_noise_005/pmrf/num_flow_steps=100/xhat"),
    ("Naive Flow",                  "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_super_resolution_naive_flow_test_K100/sr_bicubic_x4_gaussian_noise_005/naive_flow/num_flow_steps=100/xhat"),
    ("Flow cond. on $Y$",           "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_super_resolution_post_con_on_y_test_K100/sr_bicubic_x4_gaussian_noise_005/posterior_conditioned_on_y/num_flow_steps=100/xhat"),
    ("Flow cond. on $\\hat{X}^*$",  "/content/drive/MyDrive/PMRF/outputs/zappos_data/new_zap_super_resolution_post_con_on_mmse_test_K100/sr_bicubic_x4_gaussian_noise_005/posterior_conditioned_on_mmse/num_flow_steps=100/xhat"),
]

# Which samples to show, left to right.
#   - None  -> auto-pick the first N_COLUMNS samples common to every folder
#   - list  -> explicit filenames; N_COLUMNS is then ignored
FILENAMES = [
    "8142128.282626_30830.jpg",
    "8140291.412799_41502.jpg",
    "8132538.3_15542.jpg",
    "8088749.395751_06760.jpg",
    "8084611.20_34260.jpg",
    "8081317.399_27845.jpg",
    "8077306.11414_44557.jpg",
    "8076793.59416_42318.jpg",
]
N_COLUMNS = 8

OUTFILE   = "grid_zappos_super_resolution.pdf"
TITLE     = None          # leave None: the LaTeX \caption carries the title
LABEL_IN  = 0.42          # width of the rotated-label margin, in inches
CELL_IN   = 1.5           # size of one image cell, in inches (cells stay square)
FONTSIZE  = 10            # row-label size; lower it if a long label overruns its row
ROW_GAP_IN = 0.055        # vertical gap between rows, in inches (0 = touching)
COL_GAP_IN = 0.0          # horizontal gap between columns (0 = columns butt together)

# ======================================================================


def _use_latex_like_font():
    for f in [
        "/usr/share/texmf/fonts/opentype/public/lm/lmroman10-regular.otf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf",
    ]:
        if os.path.exists(f):
            try:
                fm.fontManager.addfont(f)
            except Exception:
                pass
    plt.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Latin Modern Roman", "DejaVu Serif"],
        "mathtext.fontset": "cm",
    })


def _list_images(folder):
    exts = (".png", ".jpg", ".jpeg", ".bmp", ".webp")
    if not os.path.isdir(folder):
        raise FileNotFoundError(f"Folder does not exist: {folder}")
    return sorted(f for f in os.listdir(folder) if f.lower().endswith(exts))


def _stem(name):
    return os.path.splitext(name)[0]


def pick_filenames(rows, n):
    """Filenames present in every folder, matched on the stem (ignores extension)."""
    per_folder = []
    for label, folder in rows:
        files = _list_images(folder)
        if not files:
            raise RuntimeError(f"No images found in: {folder}")
        per_folder.append({_stem(f): f for f in files})

    common = set(per_folder[0])
    for d in per_folder[1:]:
        common &= set(d)
    if not common:
        raise RuntimeError(
            "No filenames are common to all folders. Check that the same test "
            "images were written by every run, and that extensions match."
        )
    chosen = sorted(common)[:n]
    if len(chosen) < n:
        print(f"[warn] only {len(chosen)} common samples available, using those.")
    # map back to the real filename inside each folder
    return [[per_folder[i][s] for s in chosen] for i in range(len(rows))], chosen


def resolve_filenames(rows, filenames):
    """Use explicit filenames; fall back to stem matching if an extension differs."""
    resolved = []
    for label, folder in rows:
        by_stem = {_stem(f): f for f in _list_images(folder)}
        got = []
        for fn in filenames:
            if fn in by_stem.values():
                got.append(fn)
            elif _stem(fn) in by_stem:
                got.append(by_stem[_stem(fn)])
            else:
                raise FileNotFoundError(f"'{fn}' not found in {folder}")
        resolved.append(got)
    return resolved, filenames


def build_grid(rows=ROWS, filenames=FILENAMES, n_columns=N_COLUMNS,
               outfile=OUTFILE, title=TITLE, label_in=LABEL_IN,
               cell_in=CELL_IN, fontsize=FONTSIZE,
               row_gap_in=ROW_GAP_IN, col_gap_in=COL_GAP_IN):
    """Geometry is computed in inches so that every image cell stays square."""
    _use_latex_like_font()

    if filenames is None:
        per_row_files, chosen = pick_filenames(rows, n_columns)
    else:
        per_row_files, chosen = resolve_filenames(rows, filenames)
        n_columns = len(chosen)

    n_rows = len(rows)
    title_in = 0.30 if title else 0.0

    grid_w_in = n_columns * cell_in + (n_columns - 1) * col_gap_in
    grid_h_in = n_rows * cell_in + (n_rows - 1) * row_gap_in
    fig_w = label_in + grid_w_in
    fig_h = grid_h_in + title_in
    fig = plt.figure(figsize=(fig_w, fig_h))

    # inch -> figure-fraction helpers
    fx = lambda v: v / fig_w
    fy = lambda v: v / fig_h

    for r, (label, folder) in enumerate(rows):
        # top edge of this row, measured in inches from the bottom
        row_top_in = grid_h_in - r * (cell_in + row_gap_in)
        y_mid = fy(row_top_in - cell_in / 2.0)
        fig.text(fx(label_in * 0.5), y_mid, label,
                 rotation=90, va="center", ha="center", fontsize=fontsize)
        for c in range(n_columns):
            path = os.path.join(folder, per_row_files[r][c])
            img = Image.open(path).convert("RGB")
            left_in = label_in + c * (cell_in + col_gap_in)
            ax = fig.add_axes([fx(left_in), fy(row_top_in - cell_in),
                               fx(cell_in), fy(cell_in)])
            ax.imshow(img, aspect="auto")     # cell is square, so no distortion
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_visible(False)

    if title:
        fig.text(fx(label_in + grid_w_in / 2.0), fy(fig_h - 0.09), title,
                 ha="center", va="top", fontsize=fontsize + 1.5)

    fig.savefig(outfile, bbox_inches="tight", pad_inches=0.01, dpi=300)
    plt.close(fig)
    print(f"wrote {outfile}   ({n_rows} rows x {n_columns} columns)")
    print("samples used:", ", ".join(chosen))
    return outfile


if __name__ == "__main__":
    build_grid()

wrote grid_zappos_super_resolution.pdf   (7 rows x 8 columns)
samples used: 8142128.282626_30830.jpg, 8140291.412799_41502.jpg, 8132538.3_15542.jpg, 8088749.395751_06760.jpg, 8084611.20_34260.jpg, 8081317.399_27845.jpg, 8077306.11414_44557.jpg, 8076793.59416_42318.jpg


In [ ]:
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from PIL import Image

# =============================== CONFIG ===============================

# Order of the four rows inside each dataset block, and the key each row
# looks up in the "folders" dict below.
ROW_ORDER = [
    ("Ground truth",                "gt"),
    ("Degraded $Y$",                "y"),
    ("Posterior mean $\\hat{X}^*$", "mmse"),
    ("PMRF",                        "pmrf"),
]

# Order of the three column groups, and the title printed above each.
TASK_ORDER  = ["colorization", "denoising", "sr"]
TASK_TITLES = {
    "colorization": "Colorization",
    "denoising":    "Gaussian denoising",
    "sr":           "$4\\times$ super-resolution",
}

IMNET = "/content/drive/MyDrive/PMRF/outputs/imagenet_data"
ZAP   = "/content/drive/MyDrive/PMRF/outputs/zappos_data"

BLOCKS = [
    {
        "name": "ImageNet-mini",
        "tasks": {
            "colorization": {
                "folders": {
                    "gt":   "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test",
                    "y":    f"{IMNET}/new_colorization_pmrf_test_K100/colorization_gaussian_noise_005/pmrf/y",
                    "mmse": f"{IMNET}/new_colorization_mmse_test_K100/colorization_gaussian_noise_005/mmse/xhat",
                    "pmrf": f"{IMNET}/new_colorization_pmrf_test_K100/colorization_gaussian_noise_005/pmrf/num_flow_steps=100/xhat",
                },
                "samples": ["ILSVRC2012_val_00000013.JPEG",
                            "ILSVRC2012_val_00004700.JPEG",
                            "ILSVRC2012_val_00011488.JPEG"],
            },
            "denoising": {
                "folders": {
                    "gt":   "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test",
                    "y":    f"{IMNET}/new_gaussian_noise_pmrf_test_K100/gaussian_noise_02/pmrf/y",
                    "mmse": f"{IMNET}/new_gaussian_noise_mmse_test_K100/gaussian_noise_02/mmse/xhat",
                    "pmrf": f"{IMNET}/new_gaussian_noise_pmrf_test_K100/gaussian_noise_02/pmrf/num_flow_steps=100/xhat",
                },
                "samples": ["ILSVRC2012_val_00000013.JPEG",
                            "ILSVRC2012_val_00004700.JPEG",
                            "ILSVRC2012_val_00011488.JPEG"],
            },
            "sr": {
                "folders": {
                    "gt":   "/content/drive/MyDrive/PMRF/data/imagenet-mini-centercropped/test",
                    "y":    f"{IMNET}/new_super_resolution_pmrf_test_K100/sr_bicubic_x4_gaussian_noise_005/pmrf/y",
                    "mmse": f"{IMNET}/new_super_resolution_mmse_test_K100/sr_bicubic_x4_gaussian_noise_005/mmse/xhat",
                    "pmrf": f"{IMNET}/new_super_resolution_pmrf_test_K100/sr_bicubic_x4_gaussian_noise_005/pmrf/num_flow_steps=100/xhat",
                },
                "samples": ["ILSVRC2012_val_00000013.JPEG",
                            "ILSVRC2012_val_00004700.JPEG",
                            "ILSVRC2012_val_00011488.JPEG"],
            },
        },
    },
    {
        "name": "UT Zappos50K",
        "tasks": {
            "colorization": {
                "folders": {
                    "gt":   "/content/drive/MyDrive/PMRF/data/zap50k_128/test",
                    "y":    f"{ZAP}/new_zap_colorization_pmrf_test_K100/colorization_gaussian_noise_005/pmrf/y",
                    "mmse": f"{ZAP}/new_zap_colorization_mmse_test_K100/colorization_gaussian_noise_005/mmse/xhat",
                    "pmrf": f"{ZAP}/new_zap_colorization_pmrf_test_K100/colorization_gaussian_noise_005/pmrf/num_flow_steps=100/xhat",
                },
                "samples": ["8159063.309554_03476.jpg", "8154575.910_23464.jpg", "8129141.3298_40653"],
            },
            "denoising": {
                "folders": {
                    "gt":   "/content/drive/MyDrive/PMRF/data/zap50k_128/test",
                    "y":    f"{ZAP}/new_zap_gaussian_noise_pmrf_test_K100/gaussian_noise_02/pmrf/y",
                    "mmse": f"{ZAP}/new_zap_gaussian_noise_mmse_test_K100/gaussian_noise_02/mmse/xhat",
                    "pmrf": f"{ZAP}/new_zap_gaussian_noise_pmrf_test_K100/gaussian_noise_02/pmrf/num_flow_steps=100/xhat",
                },
                "samples": ["8159063.309554_03476.jpg", "8154575.910_23464.jpg", "8129141.3298_40653"],
            },
            "sr": {
                "folders": {
                    "gt":   "/content/drive/MyDrive/PMRF/data/zap50k_128/test",
                    "y":    f"{ZAP}/new_zap_super_resolution_pmrf_test_K100/sr_bicubic_x4_gaussian_noise_005/pmrf/y",
                    "mmse": f"{ZAP}/new_zap_super_resolution_mmse_test_K100/sr_bicubic_x4_gaussian_noise_005/mmse/xhat",
                    "pmrf": f"{ZAP}/new_zap_super_resolution_pmrf_test_K100/sr_bicubic_x4_gaussian_noise_005/pmrf/num_flow_steps=100/xhat",
                },
                "samples": ["8159063.309554_03476.jpg", "8154575.910_23464.jpg", "8129141.3298_40653"],
            },
        },
    },
]

OUTFILE       = "grid_compact_additional_domains.pdf"
CELL_IN       = 1.35   # size of one image cell, inches (cells stay square)
ROW_LABEL_IN  = 0.40   # inner margin: row labels
BLOCK_LABEL_IN= 0.36   # outer margin: dataset labels (set 0.0 to drop them)
TITLE_IN      = 0.34   # top strip: column-group titles
ROW_GAP_IN    = 0.045  # gap between rows inside a block
BLOCK_GAP_IN  = 0.20   # extra gap between the two dataset blocks
GROUP_GAP_IN  = 0.10   # extra gap between column groups
FONTSIZE      = 9.5
TITLE_FONTSIZE= 10.5
BLOCK_FONTSIZE= 10.5

# ======================================================================


def _font():
    for f in ["/usr/share/texmf/fonts/opentype/public/lm/lmroman10-regular.otf",
              "/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf"]:
        if Path(f).exists():
            try:
                fm.fontManager.addfont(f)
            except Exception:
                pass
    plt.rcParams.update({"font.family": "serif",
                         "font.serif": ["Latin Modern Roman", "DejaVu Serif"],
                         "mathtext.fontset": "cm"})


def _resolve(folder, name):
    folder = Path(folder)
    if not folder.is_dir():
        raise FileNotFoundError(f"folder does not exist: {folder}")
    exact = folder / name
    if exact.exists():
        return exact
    hits = list(folder.glob(Path(name).stem + ".*"))
    if not hits:
        raise FileNotFoundError(f"'{name}' not found in {folder}")
    return hits[0]


def build(blocks=BLOCKS, outfile=OUTFILE, cell_in=CELL_IN,
          row_label_in=ROW_LABEL_IN, block_label_in=BLOCK_LABEL_IN,
          title_in=TITLE_IN, row_gap_in=ROW_GAP_IN,
          block_gap_in=BLOCK_GAP_IN, group_gap_in=GROUP_GAP_IN,
          fontsize=FONTSIZE, title_fontsize=TITLE_FONTSIZE,
          block_fontsize=BLOCK_FONTSIZE):
    _font()

    n_groups = len(TASK_ORDER)
    per_group = [len(blocks[0]["tasks"][t]["samples"]) for t in TASK_ORDER]
    n_col = sum(per_group)
    n_rows_block = len(ROW_ORDER)
    n_blocks = len(blocks)

    # --- geometry, all in inches so cells stay square --------------------
    grid_w = n_col * cell_in + (n_groups - 1) * group_gap_in
    block_h = n_rows_block * cell_in + (n_rows_block - 1) * row_gap_in
    grid_h = n_blocks * block_h + (n_blocks - 1) * block_gap_in

    left_in = block_label_in + row_label_in
    fig_w = left_in + grid_w
    fig_h = grid_h + title_in
    fig = plt.figure(figsize=(fig_w, fig_h))
    fx, fy = lambda v: v / fig_w, lambda v: v / fig_h

    # x offset of each column, accounting for the gaps between groups
    col_x, x = [], left_in
    for gi, t in enumerate(TASK_ORDER):
        for _ in range(per_group[gi]):
            col_x.append(x)
            x += cell_in
        if gi < n_groups - 1:
            x += group_gap_in

    # --- column-group titles --------------------------------------------
    k = 0
    for gi, t in enumerate(TASK_ORDER):
        x0 = col_x[k]
        x1 = col_x[k + per_group[gi] - 1] + cell_in
        fig.text(fx((x0 + x1) / 2), fy(grid_h + title_in * 0.32),
                 TASK_TITLES[t], ha="center", va="bottom",
                 fontsize=title_fontsize)
        k += per_group[gi]

    # --- image rows ------------------------------------------------------
    for bi, block in enumerate(blocks):
        block_top = grid_h - bi * (block_h + block_gap_in)

        if block_label_in > 0:
            fig.text(fx(block_label_in * 0.5), fy(block_top - block_h / 2),
                     block["name"], rotation=90, va="center", ha="center",
                     fontsize=block_fontsize)

        for r, (label, key) in enumerate(ROW_ORDER):
            row_top = block_top - r * (cell_in + row_gap_in)
            fig.text(fx(block_label_in + row_label_in * 0.5),
                     fy(row_top - cell_in / 2), label,
                     rotation=90, va="center", ha="center", fontsize=fontsize)

            c = 0
            for t in TASK_ORDER:
                spec = block["tasks"][t]
                for name in spec["samples"]:
                    img = Image.open(_resolve(spec["folders"][key], name)).convert("RGB")
                    ax = fig.add_axes([fx(col_x[c]), fy(row_top - cell_in),
                                       fx(cell_in), fy(cell_in)])
                    ax.imshow(img, aspect="auto")
                    ax.set_xticks([]); ax.set_yticks([])
                    for sp in ax.spines.values():
                        sp.set_visible(False)
                    c += 1

    fig.savefig(outfile, bbox_inches="tight", pad_inches=0.01, dpi=300)
    plt.close(fig)
    print(f"wrote {outfile}  ({n_blocks * n_rows_block} rows x {n_col} columns)")
    return outfile


if __name__ == "__main__":
    build()

wrote grid_compact_additional_domains.pdf  (8 rows x 9 columns)


In [ ]:
"""
Chapter 5.1 --- blind face restoration grid.

Four rows (ground truth / degraded / posterior mean / PMRF) over six manually
chosen CelebA-Test samples.  Rotated row labels, seamless columns, square cells.

Fill in the four folders and the six filenames, then run.
"""

from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from PIL import Image

# =============================== CONFIG ===============================

ROWS = [
    ("Ground truth",                "/content/drive/MyDrive/PMRF/data/celeba_512_validation"),
    ("Degraded $Y$",                "/content/drive/MyDrive/PMRF/data/celeba_512_validation_lq"),
    ("Posterior mean $\\hat{X}^*$", "/content/drive/MyDrive/PMRF/outputs/papers_data/celeba512_pmrf/restored_images_posterior_mean"),
    ("PMRF",                        "/content/drive/MyDrive/PMRF/outputs/papers_data/celeba512_pmrf/restored_images"),
]

# The six samples you pick, left to right. Extensions may differ between
# folders; matching is done on the filename stem.
FILENAMES = [
    "00000002.png",
    "00000005.png",
    "00000026.png",
    "00000007.png",
    "00000011.png",
    "00000015.png"
]

OUTFILE    = "grid_blind_face.pdf"
LABEL_IN   = 0.42     # width of the rotated-label margin, inches
CELL_IN    = 1.6      # size of one image cell, inches (cells stay square)
FONTSIZE   = 10
ROW_GAP_IN = 0.055    # vertical gap between rows (0 = touching)
COL_GAP_IN = 0.0      # horizontal gap between columns (0 = butting together)

# ======================================================================


def _font():
    for f in ["/usr/share/texmf/fonts/opentype/public/lm/lmroman10-regular.otf",
              "/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf"]:
        if Path(f).exists():
            try:
                fm.fontManager.addfont(f)
            except Exception:
                pass
    plt.rcParams.update({"font.family": "serif",
                         "font.serif": ["Latin Modern Roman", "DejaVu Serif"],
                         "mathtext.fontset": "cm"})


def _resolve(folder, name):
    """Find `name` in `folder`, tolerating a different file extension."""
    folder = Path(folder)
    if not folder.is_dir():
        raise FileNotFoundError(f"folder does not exist: {folder}")
    exact = folder / name
    if exact.exists():
        return exact
    hits = list(folder.glob(Path(name).stem + ".*"))
    if not hits:
        raise FileNotFoundError(f"'{name}' not found in {folder}")
    return hits[0]


def build(rows=ROWS, filenames=FILENAMES, outfile=OUTFILE,
          label_in=LABEL_IN, cell_in=CELL_IN, fontsize=FONTSIZE,
          row_gap_in=ROW_GAP_IN, col_gap_in=COL_GAP_IN):
    _font()
    n_row, n_col = len(rows), len(filenames)

    grid_w = n_col * cell_in + (n_col - 1) * col_gap_in
    grid_h = n_row * cell_in + (n_row - 1) * row_gap_in
    fig_w, fig_h = label_in + grid_w, grid_h
    fig = plt.figure(figsize=(fig_w, fig_h))
    fx, fy = lambda v: v / fig_w, lambda v: v / fig_h

    for r, (label, folder) in enumerate(rows):
        top_in = grid_h - r * (cell_in + row_gap_in)
        fig.text(fx(label_in * 0.5), fy(top_in - cell_in / 2), label,
                 rotation=90, va="center", ha="center", fontsize=fontsize)
        for c, name in enumerate(filenames):
            img = Image.open(_resolve(folder, name)).convert("RGB")
            ax = fig.add_axes([fx(label_in + c * (cell_in + col_gap_in)),
                               fy(top_in - cell_in), fx(cell_in), fy(cell_in)])
            ax.imshow(img, aspect="auto")
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_visible(False)

    fig.savefig(outfile, bbox_inches="tight", pad_inches=0.01, dpi=300)
    plt.close(fig)
    print(f"wrote {outfile}  ({n_row} rows x {n_col} columns)")
    return outfile


if __name__ == "__main__":
    build()

wrote grid_blind_face.pdf  (4 rows x 6 columns)
